In [1]:
import pandas as pd

PATH_GAMLOGS_COMBINED = '../data/all_gamelogs_combined.csv'
PATH_TO_MODEL_dir = '../models/threes/'

### update data

In [6]:
import pandas as pd
import pandas as pd
import os

datas = []
def min_played_minutes(row):
    """Convert 'mm:ss' string to total minutes as float."""
    if pd.isna(row):
        return 0.0
    try:
        minutes, seconds = map(int, row.split(':'))
        total_minutes = minutes + seconds / 60.0
        return total_minutes
    except Exception as e:
        print(f"Error parsing mp '{row}': {e}")
        return 0.0
    


for dir,_,files in os.walk("../data/gamelogs/"):
    for file in files:
        if file.endswith(".csv"):
            path = os.path.join(dir, file)
            print(f"Processing {path}")
            df = pd.read_csv(path)
            datas.append(df)


data_ori =pd.concat(datas, ignore_index=True)
print(f"Total gamelog rows combined: {len(data_ori)}")
drop_cols = ["gs", "home_away", "href", 'result']
data = data_ori.copy()
data.dropna(subset=["mp"], inplace=True)
data['is_home'] =data["home_away"].apply(lambda x: 1 if x == "@" else 0)
data["is_win"] = data["result"].apply(lambda x: 1 if x.startswith("W") else 0)
data["date"] = pd.to_datetime(data["date"])
data["mp_minutes"] = data["mp"].apply(min_played_minutes)
data['usage'] = data['mp_minutes'] / 48.0
data.drop(drop_cols, axis=1, inplace=True)
data.drop_duplicates(inplace=True, subset=['player', 'date', 'season'], keep='last')
data.to_csv("../data/all_gamelogs_combined.csv", index=False)
print("the lastest date in the data is:", data['date'].max())
print("the total number of unique players is:", data['player'].nunique())
print("the total number of gamelog rows is:", len(data))

Processing ../data/gamelogs/gamelogs_2021.csv
Processing ../data/gamelogs/gamelogs_2022.csv
Processing ../data/gamelogs/gamelogs_2023.csv
Processing ../data/gamelogs/gamelogs_2024.csv
Processing ../data/gamelogs/gamelogs_2025.csv
Processing ../data/gamelogs/gamelogs_2026.csv
Total gamelog rows combined: 87503
the lastest date in the data is: 2026-01-29 00:00:00
the total number of unique players is: 802
the total number of gamelog rows is: 47502


In [ ]:
def build_3p_defense_ranks(gamelog_df: pd.DataFrame) -> pd.DataFrame:
    """
    Builds season-level 3P defense rankings from game logs.
    Lower rank = better defense.
    """

    df = gamelog_df.copy()

    print(df.columns)
    # --- Aggregate opponent shooting by defensive team ---
    season_def = (
        df.groupby(["season", "team"])
          .agg(
              games_played=("game_date", "count"),
              opp_fg3a=("opp_fg3a", "sum"),
              opp_fg3m=("opp_fg3m", "sum"),
          )
          .reset_index()
    )

    # --- Per-game metrics ---
    season_def["opp_fg3a_pg"] = season_def["opp_fg3a"] / season_def["games_played"]
    season_def["opp_fg3m_pg"] = season_def["opp_fg3m"] / season_def["games_played"]
    season_def["opp_fg3_pct"] = (
        season_def["opp_fg3m"] / season_def["opp_fg3a"]
    ).fillna(0)

    # --- Ranks (lower = better defense) ---
    season_def["rank_fg3m_pg"] = season_def.groupby("season")["opp_fg3m_pg"] \
        .rank(method="min", ascending=True)

    season_def["rank_fg3a_pg"] = season_def.groupby("season")["opp_fg3a_pg"] \
        .rank(method="min", ascending=True)

    season_def["rank_fg3_pct"] = season_def.groupby("season")["opp_fg3_pct"] \
        .rank(method="min", ascending=True)

    # --- Composite score ---
    season_def["def_3p_score"] = (
        0.5 * season_def["rank_fg3m_pg"] +
        0.3 * season_def["rank_fg3a_pg"] +
        0.2 * season_def["rank_fg3_pct"]
    )

    # --- Final defensive position ---
    season_def["def_3p_rank"] = season_def.groupby("season")["def_3p_score"] \
        .rank(method="min", ascending=True).astype(int)

    # --- Final output ---
    return season_def[[
        "season",
        "team",
        "def_3p_rank",
        "opp_fg3a_pg",
        "opp_fg3m_pg",
        "opp_fg3_pct"
    ]].sort_values(["season", "def_3p_rank"])


In [12]:
data.head(2)

,player,season,date,team,opp,mp,fg,fga,fg3,fg3a,...,ast,stl,blk,tov,pf,pts,is_home,is_win,mp_minutes,usage
0,Nathan Knight,2021,2020-12-23,ATL,CHI,05:04,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,2.0,2.0,0.0,1,1,5.066667,0.105556
1,Nathan Knight,2021,2020-12-26,ATL,MEM,08:35,4.0,5.0,2.0,3.0,...,0.0,0.0,0.0,1.0,1.0,14.0,1,1,8.583333,0.178819


In [13]:
build_3p_defense_ranks(data)

Index(['player', 'season', 'date', 'team', 'opp', 'mp', 'fg', 'fga', 'fg3',
       'fg3a', 'ft', 'fta', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov',
       'pf', 'pts', 'is_home', 'is_win', 'mp_minutes', 'usage'],
      dtype='object')


KeyError: "Column(s) ['game_date', 'opp_fg3a', 'opp_fg3m'] do not exist"

In [7]:
import joblib
from model_training.config import PATH_TO_MODEL_dir
joblib.load(PATH_TO_MODEL_dir + "features.joblib")

ModuleNotFoundError: No module named 'model_training'

In [5]:
import pandas as pd

def build_features(data):
    df = data.copy()
    df = df.sort_values(["player","date"])
    df["date"] = pd.to_datetime(df["date"])

    df["min"] = df["mp_minutes"].replace(0, np.nan)  # prevent divide-by-zero
    df["fg3a_per_min"] = df["fg3a"] / df["min"]

    df["fga"] = df["fga"].replace(0, np.nan)
    df["fg3a_share"] = df["fg3a"] / df["fga"]

    df["usage_proxy"] = (df["fga"] + 0.44 * df["fta"]) / df["min"]

    df["minutes_rolling_5"] = (
        df.groupby("player")["min"]
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["starter_flag"] = (df["min"] >= 24).astype(int)

    group = df.groupby("player")

    df["fg3_rolling_3"] = group["fg3"].rolling(3).mean().reset_index(level=0, drop=True)
    df["fg3_rolling_5"] = group["fg3"].rolling(5).mean().reset_index(level=0, drop=True)
    df["fg3a_rolling_5"] = group["fg3a"].rolling(5).mean().reset_index(level=0, drop=True)

    df["fg3_pct_rolling_10"] = (
        group["fg3"].rolling(10).sum().reset_index(level=0, drop=True) /
        group["fg3a"].rolling(10).sum().reset_index(level=0, drop=True)
    )

    df["fg3_std_rolling_10"] = group["fg3"].rolling(10).std().reset_index(level=0, drop=True)
    df["minutes_rolling_5"] = group["min"].rolling(5).mean().reset_index(level=0, drop=True)
    df["minutes_std_rolling_10"] = group["min"].rolling(10).std().reset_index(level=0, drop=True)

    season_group = df.groupby(["player", "season"])

    df["fg3_pct_season"] = season_group["fg3"].transform("sum") / season_group["fg3a"].transform("sum")
    df["fg3_over_expected"] = df["fg3_pct_rolling_10"] - df["fg3_pct_season"]

    # -------------------------------
    # EFFICIENCY
    # -------------------------------
    df["pts_per_fga"] = df["pts"] / df["fga"]

    # -------------------------------
    # GAME CONTEXT

    df["home_game"] = df["is_home"].astype(int)
    df = df.sort_values(["player", "date"])
    df["days_rest"] = (
        df.groupby("player")["date"]
        .diff()
        .dt.days
    )
    df["back_to_back"] = (df["days_rest"] == 1).astype(int)

    df["opp_fg3_allowed_avg"] = df.groupby("opp")["fg3"].transform("mean")
    df["opp_pace_proxy"] = df.groupby("opp")["fga"].transform("mean")

    df["team_fg3_rate"] = df.groupby("team")["fg3a"].transform("mean")

    FEATURE_COLS = [
        "fg3a_per_min", "fg3a_share", "usage_proxy",
        "fg3_rolling_3", "fg3_rolling_5", "fg3a_rolling_5",
        "fg3_pct_rolling_10", "fg3_pct_season", "fg3_over_expected",
        "fg3_std_rolling_10", "minutes_rolling_5", "minutes_std_rolling_10",
        "pts_per_fga", "home_game", "starter_flag",
        "days_rest", "back_to_back",
        "opp_fg3_allowed_avg", "opp_pace_proxy", "team_fg3_rate"
    ]



    return df[FEATURE_COLS + ["fg3"]].dropna()

    


In [102]:
def build_features_no_leak(data):
    df = data.copy()
    df = df.sort_values(["player", "date"]).copy()
    g = df.groupby("player")

    df["date"] = pd.to_datetime(df["date"])



    # Past-only versions of stats
    df["min_prev"]  = g["mp_minutes"].shift(1)
    df["fga_prev"]  = g["fga"].shift(1)
    df["fg3a_prev"] = g["fg3a"].shift(1)
    df["pts_prev"]  = g["pts"].shift(1)

    # Rolling means based on PREVIOUS games only
    df["min_rolling_5"]  = g["mp_minutes"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)
    df["fga_rolling_5"]  = g["fga"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)
    df["fg3a_rolling_5"] = g["fg3a"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)

    df["fg3_rolling_5"]  = g["fg3"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)

    # Form shooting % (past only)
    made10 = g["fg3"].shift(1).rolling(10).sum().reset_index(level=0, drop=True)
    att10  = g["fg3a"].shift(1).rolling(10).sum().reset_index(level=0, drop=True)
    df["fg3_pct_rolling_10"] = made10 / att10

    # Days rest (past-only is naturally safe)
    df["days_rest"] = g["date"].diff().dt.days
    df["back_to_back"] = (df["days_rest"] == 1).astype(int)

    # Home/starter flags
    if "is_home" in df.columns:
        df["home_game"] = df["is_home"].astype(int)
    elif "home" in df.columns:
        df["home_game"] = df["home"].astype(int)

    df["starter_flag"] = (df["mp_minutes"] >= 24).astype(int)

    return df


In [103]:
def add_player_baselines(data):
    df = data.copy()
    df = df.sort_values(["player", "date"]).copy()
    g = df.groupby("player")

    # Avg 3PA up to yesterday
    df["player_fg3a_season_avg"] = (
        g["fg3a"]
        .shift(1)
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Career/season 3P% up to yesterday
    made = g["fg3"].shift(1).expanding().sum()
    att  = g["fg3a"].shift(1).expanding().sum()

    df["player_fg3_pct_season"] = (made / att).reset_index(level=0, drop=True)

    # Avg minutes
    df["player_min_season_avg"] = (
        g["mp_minutes"]
        .shift(1)
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Avg usage
    df["player_usage_season"] = (
        g["usage"]
        .shift(1)
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
    )

    df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


    return df


In [104]:
data = pd.read_csv(PATH_GAMLOGS_COMBINED)
data = build_features_no_leak(data)
data = add_player_baselines(data)
data.head(1)

C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


,player,season,date,team,opp,mp,fg,fga,fg3,fg3a,...,fg3_rolling_5,fg3_pct_rolling_10,days_rest,back_to_back,home_game,starter_flag,player_fg3a_season_avg,player_fg3_pct_season,player_min_season_avg,player_usage_season
7868,A.J. Green,2023,2022-10-22,MIL,HOU,02:14,0.0,0.0,0.0,0.0,...,1.4,0.393939,NaN,0,0,0,3.258298,0.357287,21.878335,0.455799


### Features

In [105]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib

FEATURES = [
    "min_rolling_5", "fga_rolling_5", "fg3a_rolling_5",
    "fg3_rolling_5", "fg3_pct_rolling_10",
    "player_fg3a_season_avg", "player_fg3_pct_season",
    "player_min_season_avg", "player_usage_season",
    "home_game", "days_rest", "back_to_back", "starter_flag",
]
joblib.dump(FEATURES, PATH_TO_MODEL_dir + "fg3_features.joblib")



['../models/threes/fg3_features.joblib']

### Ridge

#### 3fg

In [106]:
df = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])

# build train features
df = build_features_no_leak(df)
df = add_player_baselines(df)

train_df = df.dropna(subset=FEATURES + ["fg3"]).copy()
X, y = train_df[FEATURES], train_df["fg3"]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
pipe.fit(X, y)

joblib.dump(pipe, PATH_TO_MODEL_dir + "fg3_pipe_rid_test.joblib")


C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


['../models/threes/fg3_pipe_rid_test.joblib']

### rate

In [107]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib
import numpy as np

# df already has features added
train_df = df.dropna(subset=FEATURES + ["fg3", "fg3a"]).copy()
train_df = train_df[train_df["fg3a"] > 0]

X = train_df[FEATURES]
y_rate = train_df["fg3"] / train_df["fg3a"]

rate_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

rate_pipe.fit(X, y_rate)
joblib.dump(rate_pipe, PATH_TO_MODEL_dir + "fg3_rate_pipe_rid1.joblib")

['../models/threes/fg3_rate_pipe_rid1.joblib']

#### hist gradient

In [108]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
import joblib

# df already has features added
train_df = df.dropna(subset=FEATURES + ["fg3", "fg3a"]).copy()
train_df = train_df[train_df["fg3a"] > 0]

X = train_df[FEATURES]
y = train_df["fg3"]


pipe = Pipeline([
    ("model", HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        random_state=42
    ))
])

pipe.fit(X, y)

joblib.dump(pipe, PATH_TO_MODEL_dir + "fg3_pipe_his.joblib")


['../models/threes/fg3_pipe_his.joblib']

In [109]:
import joblib
import numpy as np
import pandas as pd

def predict_fg3_1(
    history_df: pd.DataFrame,
    player: str,
    date,
    team: str,
    opp: str,
    is_home: int | None = None,
    starter: int | None = None,
    pipe_path: str = PATH_TO_MODEL_dir + "fg3_pipe_rid1.joblib",
    features_path: str = PATH_TO_MODEL_dir + "fg3_features.joblib",
) -> float:
    pipe = joblib.load(pipe_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # last known game for that player
    ph = history[history["player"] == player].sort_values("date")
    if ph.empty:
        raise ValueError(f"No history found for player={player}")

    prev = ph.iloc[-1]

    # default is_home/starter to previous line if not provided
    if is_home is None:
        is_home = int(prev.get("is_home", 0))
    if starter is None:
        starter = int(prev.get("starter_flag", 1))

    # "fill the rest with prev line"
    today_row = {
        "player": player,
        "season": prev["season"],          # carry forward (or set explicitly if you want)
        "date": pd.Timestamp(date),
        "team": team,
        "opp": opp,

        # previous-line filled fields (so nothing is hard-coded)
        "mp_minutes": float(prev["mp_minutes"]),
        "fga": float(prev["fga"]),
        "fg3a": float(prev["fg3a"]),
        "pts": float(prev["pts"]),
        "usage": float(prev["usage"]),

        "is_home": int(is_home),
        "starter_flag": int(starter),

        # target unknown
        "fg3": np.nan,
    }

    today_df = pd.DataFrame([today_row])

    # append + rebuild features (rolling & baselines will use past games via shift(1))
    combined = pd.concat([history, today_df], ignore_index=True)

    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    X_today = combined.iloc[-1:][FEATURES]

    # if any features still NaN (early season / not enough games), you can fill them:
    # X_today = X_today.fillna(X_today.mean(numeric_only=True))

    pred = float(pipe.predict(X_today)[0])
    return pred


In [110]:
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])

pipe_path = PATH_TO_MODEL_dir + "fg3_pipe_his.joblib"

pred = predict_fg3_1(
    history_df=history,
    player="Stephen Curry",
    date="2026-01-23",
    team="GSW",
    opp="LAL",
    is_home=1,
    starter=1,
    pipe_path=pipe_path,
)

print(pred)


0.8082795754574491


C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


In [111]:
import pandas as pd
import numpy as np
import joblib

def fill_expected_from_recent(ph, n=5):
    return {
        "mp_minutes": float(ph["mp_minutes"].tail(n).mean()),
        "fga": float(ph["fga"].tail(n).mean()),
        "fg3a": float(ph["fg3a"].tail(n).mean()),
        "pts": float(ph["pts"].tail(n).mean()),
        "usage": float(ph["usage"].tail(n).mean()),
    }

def predict_game_fg3_2(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    pipe_path: str = PATH_TO_MODEL_dir + "fg3_pipe_his.joblib",
    features_path: str = PATH_TO_MODEL_dir + "fg3_features.joblib",
    min_games_required: int = 10,  # recommend 10 for stable rollings
    recent_n: int = 5,             # how many games for expected line
):
    pipe = joblib.load(pipe_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # latest team per player
    latest_team = (
        history.sort_values("date")
               .groupby("player")
               .tail(1)[["player", "team"]]
    )

    game_players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    # require enough games
    game_counts = history.groupby("player").size()
    game_players = [p for p in game_players if game_counts.get(p, 0) >= min_games_required]

    rows = []
    for player in game_players:
        ph = history[history["player"] == player].sort_values("date")
        prev = ph.iloc[-1]
        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        expected = fill_expected_from_recent(ph, n=recent_n)

        rows.append({
            "player": player,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": expected["mp_minutes"],
            "fga": expected["fga"],
            "fg3a": expected["fg3a"],
            "pts": expected["pts"],
            "usage": expected["usage"],

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No players found for that matchup with sufficient history.")

    combined = pd.concat([history, today_df], ignore_index=True)

    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    # feature rows correspond to the appended today_df rows
    # feature rows correspond to the appended today_df rows
    X_today = combined.tail(len(today_df))[FEATURES].copy()

    # ✅ align indexes so mask works on both X_today and today_df
    X_today = X_today.reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # require key rollings present (avoid imputer flattening)
    min_required = ["fg3a_rolling_5", "fg3_pct_rolling_10", "min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1).to_numpy()  # numpy mask avoids index alignment issues

    X_ok = X_today.loc[mask]
    out = today_df.loc[mask, ["player", "team", "opp", "is_home"]].copy()

    preds = pipe.predict(X_ok)
    out["pred_fg3"] = preds

    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out, X_today


In [112]:
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])

away, home = ("BOS", "BRK")
pipe = PATH_TO_MODEL_dir + "fg3_pipe_his.joblib"
preds = predict_game_fg3_2(history, away, home, pipe_path=pipe, game_date="2026-01-23")
pred = preds[0]
print(pred.head(50))



                player team  opp  is_home  pred_fg3
0          Josh Minott  BOS  BRK        0  2.228328
1           Sam Hauser  BOS  BRK        0  2.225681
2           Luka Garza  BOS  BRK        0  2.133263
3   Amari Williams(TW)  BOS  BRK        0  2.074500
4        Neemias Queta  BOS  BRK        0  0.980005
5         Jayson Tatum  BOS  BRK        0  0.976970
6           JD Davison  BOS  BRK        0  0.972916
7        Drew Peterson  BOS  BRK        0  0.972488
8         Jaylen Brown  BOS  BRK        0  0.957100
9      Anfernee Simons  BOS  BRK        0  0.954513
10        Jordan Walsh  BOS  BRK        0  0.948807
11    Payton Pritchard  BOS  BRK        0  0.944792
12  Xavier Tillman Sr.  BOS  BRK        0  0.943730
13       Derrick White  BOS  BRK        0  0.914676
14       Maxwell Lewis  BRK  BOS        1  0.901191
15       Reece Beekman  BRK  BOS        1  0.871483
16       Hugo González  BOS  BRK        0  0.868278
17   Baylor Scheierman  BOS  BRK        0  0.831212
18       Chr

C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


In [113]:
from sklearn.metrics import mean_squared_error
import numpy as np

X = preds[1]

pipe = PATH_TO_MODEL_dir + "fg3_pipe_his.joblib"

pipe = joblib.load(pipe)

pred_train = pipe.predict(X)  # X from training
print("y max:", y.max())
print("pred max:", pred_train.max())
print("pred 95th:", np.quantile(pred_train, 0.95))


y max: 12.0
pred max: 2.2283277306375324
pred 95th: 2.225812906752206


In [114]:
row = X.iloc[0]
print(row.sort_values(ascending=False))
print("NaNs:", row.isna().sum())

min_rolling_5             24.836667
player_min_season_avg     21.382060
fga_rolling_5              8.200000
fg3a_rolling_5             4.200000
player_fg3a_season_avg     3.197433
days_rest                  2.000000
fg3_rolling_5              1.200000
player_usage_season        0.445460
player_fg3_pct_season      0.356661
fg3_pct_rolling_10         0.333333
home_game                  0.000000
back_to_back               0.000000
starter_flag               0.000000
Name: 0, dtype: float64
NaNs: 0


In [115]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def train_fg3_rate_model(
    csv_path=PATH_GAMLOGS_COMBINED,
    pipe_out=PATH_TO_MODEL_dir + "fg3_rate_pipe_rid2.joblib",
    features_path=PATH_TO_MODEL_dir + "fg3_features.joblib",
):
    FEATURES = joblib.load(features_path)

    df = pd.read_csv(csv_path, parse_dates=["date"])
    df = build_features_no_leak(df)
    df = add_player_baselines(df)

    # Need fg3 and fg3a to compute rate
    train_df = df.dropna(subset=FEATURES + ["fg3", "fg3a"]).copy()
    train_df = train_df[train_df["fg3a"] > 0]

    X = train_df[FEATURES]
    y_rate = train_df["fg3"] / train_df["fg3a"]   # target: 3P made per 3P attempt

    rate_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    rate_pipe.fit(X, y_rate)

    joblib.dump(rate_pipe, pipe_out)
    print(f"Saved rate model to: {pipe_out}")

# Run once:
train_fg3_rate_model()


Saved rate model to: ../models/threes/fg3_rate_pipe_rid2.joblib


C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


In [116]:
import pandas as pd
import numpy as np
import joblib

def fill_expected_from_recent(ph, n=5):
    return {
        "mp_minutes": float(ph["mp_minutes"].tail(n).mean()),
        "fga": float(ph["fga"].tail(n).mean()),
        "fg3a": float(ph["fg3a"].tail(n).mean()),
        "pts": float(ph["pts"].tail(n).mean()),
        "usage": float(ph["usage"].tail(n).mean()),
    }

def predict_game_fg3_3(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    rate_pipe_path: str = PATH_TO_MODEL_dir + "fg3_rate_pipe_rid2.joblib",   # <-- rate model
    features_path: str = PATH_TO_MODEL_dir +  "fg3_features.joblib",
    min_games_required: int = 10,
    recent_n: int = 5,
):
    rate_pipe = joblib.load(rate_pipe_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    latest_team = (
        history.sort_values("date")
               .groupby("player")
               .tail(1)[["player", "team"]]
    )

    game_players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    game_counts = history.groupby("player").size()
    game_players = [p for p in game_players if game_counts.get(p, 0) >= min_games_required]

    rows = []
    for player in game_players:
        ph = history[history["player"] == player].sort_values("date")
        prev = ph.iloc[-1]
        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        expected = fill_expected_from_recent(ph, n=recent_n)

        rows.append({
            "player": player,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": expected["mp_minutes"],
            "fga": expected["fga"],
            "fg3a": expected["fg3a"],   # <-- expected attempts
            "pts": expected["pts"],
            "usage": expected["usage"],

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No players found for that matchup with sufficient history.")

    combined = pd.concat([history, today_df], ignore_index=True)

    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    X_today = combined.tail(len(today_df))[FEATURES].copy()

    # Make indexes align (CRITICAL)
    X_today = X_today.reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    min_required = ["fg3a_rolling_5", "fg3_pct_rolling_10", "min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1)

    X_ok = X_today.loc[mask]
    out = today_df.loc[mask, ["player", "team", "opp", "is_home", "fg3a"]].copy()

    # Predict rate and convert to FG3
    pred_rate = np.clip(rate_pipe.predict(X_ok), 0, 1)
    out["pred_rate"] = pred_rate
    out["pred_fg3"] = out["fg3a"].to_numpy() * pred_rate

    out = out.drop(columns=["fg3a"])
    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out


In [117]:
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])

out = predict_game_fg3_3(
    history_df=history,
    away_team="HOU",
    home_team="DET",
    game_date="2026-01-23",
)

print(out.head(20))


               player team  opp  is_home  pred_rate  pred_fg3
0       Armoni Brooks  HOU  DET        0   0.368065  3.827879
1       Malik Beasley  DET  HOU        1   0.302443  3.266389
2          Saddiq Bey  DET  HOU        1   0.305970  2.876117
3       Reed Sheppard  HOU  DET        0   0.301600  2.231839
4     Jamorko Pickett  DET  HOU        1   0.306314  1.715356
5       Tobias Harris  DET  HOU        1   0.355146  1.420586
6          Tari Eason  HOU  DET        0   0.308125  1.417375
7       Killian Hayes  DET  HOU        1   0.297990  1.370754
8       Isaiah Livers  DET  HOU        1   0.356935  1.284965
9       Javonte Green  DET  HOU        1   0.349175  1.187196
10      Frank Jackson  DET  HOU        1   0.300671  1.082417
11      Marcus Sasser  DET  HOU        1   0.292386  1.052590
12       Anthony Lamb  HOU  DET        0   0.347793  1.043380
13        David Roddy  HOU  DET        0   0.349714  0.979198
14  Deividas Sirvydis  DET  HOU        1   0.300850  0.902549
15      

C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


In [118]:
def add_ev_columns(out: pd.DataFrame, lines=(0.5, 1.5, 2.5, 3.5)) -> pd.DataFrame:
    out = out.copy()
    for L in lines:
        out[f"edge_over_{L}"] = out["pred_fg3"] - L
    return out
out = add_ev_columns(out, lines=(0.5, 1.5, 2.5, 3.5))

In [119]:
def add_volatility(history: pd.DataFrame, out: pd.DataFrame, window=10) -> pd.DataFrame:
    h = history.copy().sort_values(["player","date"])
    vol = (
        h.groupby("player")["fg3"]
         .rolling(window)
         .std()
         .reset_index(level=0, drop=True)
    )
    h["fg3_std_recent"] = vol

    # grab latest available volatility per player
    latest = h.groupby("player").tail(1)[["player", "fg3_std_recent"]]
    out = out.merge(latest, on="player", how="left")
    return out
out = add_volatility(history, out, window=10)

In [120]:
import math

def poisson_p_ge_k(lam: float, k: int) -> float:
    """P(X >= k) for X ~ Poisson(lam)."""
    if lam <= 0:
        return 0.0
    # P(X < k) = sum_{i=0}^{k-1} e^-lam * lam^i / i!
    cdf = 0.0
    term = math.exp(-lam)
    cdf += term  # i=0
    for i in range(1, k):
        term *= lam / i
        cdf += term
    return 1.0 - cdf


In [121]:
def add_over_probabilities(out: pd.DataFrame, lines=(1.5, 2.5, 3.5)) -> pd.DataFrame:
    out = out.copy()
    for L in lines:
        k = int(L + 0.5)  # Over 2.5 -> k=3
        out[f"p_over_{L}"] = out["pred_fg3"].apply(lambda lam: poisson_p_ge_k(lam, k))
    return out
out = add_over_probabilities(out, lines=(0.5, 1.5, 2.5, 3.5))

In [122]:
def add_recommendation(out: pd.DataFrame, line=2.5,
                       min_edge=0.6, min_prob=0.58, max_std=None) -> pd.DataFrame:
    out = out.copy()
    edge_col = f"edge_over_{line}"
    prob_col = f"p_over_{line}"

    cond = (out[edge_col] >= min_edge) & (out[prob_col] >= min_prob)
    if max_std is not None and "fg3_std_recent" in out.columns:
        cond = cond & (out["fg3_std_recent"].fillna(0) <= max_std)

    out["rec"] = np.where(cond, f"OVER {line}", "PASS")
    return out


In [123]:
def enrich_predictions(history: pd.DataFrame, out: pd.DataFrame, lines=(2.5, 3.5)):
    out2 = out.copy()
    out2 = add_ev_columns(out2, lines=lines)
    out2 = add_volatility(history, out2, window=10)
    out2 = add_over_probabilities(out2, lines=lines)
    return out2


In [124]:
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])

out = predict_game_fg3_3(
    history_df=history,
    away_team="hou".upper(),
    home_team="det".upper(),
    game_date="2026-01-23",
)


C:\Users\micha\AppData\Local\Temp\ipykernel_26008\3019679672.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["player_fg3_pct_season"].fillna(df["fg3"].mean() / df["fg3a"].mean(), inplace=True)


In [125]:
out = enrich_predictions(history, out, lines=(2.5, 3.5))
out = add_recommendation(out, line=2.5, min_edge=0.6, min_prob=0.58, max_std=1.8)

In [126]:
out.sort_values("player", ascending=True, inplace=True)
out

,player,team,opp,is_home,pred_rate,pred_fg3,edge_over_2.5,edge_over_3.5,fg3_std_recent,p_over_2.5,p_over_3.5,rec
17,Aaron Holiday,HOU,DET,0,0.300986,0.842761,-1.657239,-2.657239,0.483046,0.053766,0.010817,PASS
28,Amen Thompson,HOU,DET,0,0.294940,0.353928,-2.146072,-3.146072,0.483046,0.005680,0.000493,PASS
12,Anthony Lamb,HOU,DET,0,0.347793,1.043380,-1.456620,-2.456620,1.646545,0.088451,0.021764,PASS
0,Armoni Brooks,HOU,DET,0,0.368065,3.827879,1.327879,0.327879,1.791957,0.735577,0.532202,OVER 2.5
24,Bobi Klintman,DET,HOU,1,0.295370,0.649815,-1.850185,-2.850185,0.849837,0.028322,0.004443,PASS
16,Braxton Key,DET,HOU,1,0.354695,0.851268,-1.648732,-2.648732,0.516398,0.055075,0.011187,PASS
25,Buddy Boeheim,DET,HOU,1,0.298318,0.536973,-1.963027,-2.963027,0.699206,0.017347,0.002263,PASS
15,Caris LeVert,DET,HOU,1,0.307639,0.861390,-1.638610,-2.638610,0.971825,0.056651,0.011637,PASS
30,Chaz Lanier,DET,HOU,1,0.298417,0.298417,-2.201583,-3.201583,1.100505,0.003547,0.000261,PASS
13,David Roddy,HOU,DET,0,0.349714,0.979198,-1.520802,-2.520802,0.674949,0.076515,0.017739,PASS


In [127]:
out[out["rec"] != "PASS"]

,player,team,opp,is_home,pred_rate,pred_fg3,edge_over_2.5,edge_over_3.5,fg3_std_recent,p_over_2.5,p_over_3.5,rec
0,Armoni Brooks,HOU,DET,0,0.368065,3.827879,1.327879,0.327879,1.791957,0.735577,0.532202,OVER 2.5


# 2 models

In [128]:
import math
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor


# -----------------------------
# YOU ALREADY HAVE THESE
# build_features_no_leak(df)
# add_player_baselines(df)
# -----------------------------


# -----------------------------
# 1) Helper: expected line from recent games
# -----------------------------
def recent_means(ph: pd.DataFrame, n: int = 5) -> dict:
    """Trailing averages used as the base 'expected line'."""
    return {
        "mp_minutes": float(ph["mp_minutes"].tail(n).mean()),
        "fga": float(ph["fga"].tail(n).mean()),
        "fg3a": float(ph["fg3a"].tail(n).mean()),
        "pts": float(ph["pts"].tail(n).mean()),
        "usage": float(ph["usage"].tail(n).mean()),
    }


# -----------------------------
# 2) Context multipliers for FG3A
# -----------------------------
def compute_multipliers(history: pd.DataFrame, team: str, opp: str, ph: pd.DataFrame) -> dict:
    """
    Multipliers that adjust expected FG3A:
    - pace_mult: faster environment -> more attempts
    - opp_fg3a_allowed_mult: defenses that allow more 3PA -> more attempts
    - usage_trend: rising usage -> more shots -> more 3PA
    """
    # Pace proxy: team/opp average FGA compared to league
    league_pace = history["fga"].mean()
    team_pace = history.loc[history["team"] == team, "fga"].mean()
    opp_pace = history.loc[history["opp"] == opp, "fga"].mean()

    pace_mult = ((team_pace + opp_pace) / 2) / league_pace if league_pace > 0 else 1.0
    pace_mult = float(np.clip(pace_mult, 0.85, 1.15))  # keep sane

    # Opponent 3PA allowed proxy: average fg3a by opponents facing this opp
    league_fg3a = history["fg3a"].mean()
    opp_fg3a_allowed = history.loc[history["opp"] == opp, "fg3a"].mean()
    opp_fg3a_allowed_mult = (opp_fg3a_allowed / league_fg3a) if league_fg3a > 0 else 1.0
    opp_fg3a_allowed_mult = float(np.clip(opp_fg3a_allowed_mult, 0.80, 1.25))

    # Usage trend: last5 / last10
    last5 = ph["usage"].tail(5).mean()
    last10 = ph["usage"].tail(10).mean()
    usage_trend = (last5 / last10) if (last10 and not np.isnan(last10) and last10 > 0) else 1.0
    usage_trend = float(np.clip(usage_trend, 0.85, 1.20))

    return {
        "pace_mult": pace_mult,
        "opp_fg3a_allowed_mult": opp_fg3a_allowed_mult,
        "usage_trend": usage_trend,
    }


def expected_fg3a_from_recent(history: pd.DataFrame, ph: pd.DataFrame, team: str, opp: str, recent_n: int = 5) -> float:
    """Base fg3a from trailing mean, then apply multipliers."""
    base = float(ph["fg3a"].tail(recent_n).mean())

    mults = compute_multipliers(history, team=team, opp=opp, ph=ph)
    adj = base * mults["pace_mult"] * mults["opp_fg3a_allowed_mult"] * mults["usage_trend"]

    # cap to avoid wild values: use player's recent distribution
    cap = float(ph["fg3a"].tail(20).quantile(0.95)) if len(ph) >= 20 else float(ph["fg3a"].tail(10).max())
    cap = cap if (cap and not np.isnan(cap)) else base

    return float(np.clip(adj, 0, cap))


# -----------------------------
# 3) Train two models: attempts + rate
# -----------------------------
def train_two_models(
    csv_path: str,
    features: list[str],
    attempts_pipe_path: str = "fg3a_pipe.joblib",
    rate_pipe_path: str = "fg3_rate_pipe.joblib",
):
    df = pd.read_csv(csv_path, parse_dates=["date"])
    df = build_features_no_leak(df)
    df = add_player_baselines(df)

    # ----- Attempts model: y = fg3a -----
    train_att = df.dropna(subset=features + ["fg3a"]).copy()
    X_att = train_att[features]
    y_att = train_att["fg3a"]

    attempts_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_depth=4,
            learning_rate=0.05,
            max_iter=400,
            random_state=42
        ))
    ])
    attempts_pipe.fit(X_att, y_att)

    # ----- Rate model: y = fg3 / fg3a (fg3a>0) -----
    train_rate = df.dropna(subset=features + ["fg3", "fg3a"]).copy()
    train_rate = train_rate[train_rate["fg3a"] > 0]
    X_rate = train_rate[features]
    y_rate = train_rate["fg3"] / train_rate["fg3a"]

    rate_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_depth=4,
            learning_rate=0.05,
            max_iter=400,
            random_state=42
        ))
    ])
    rate_pipe.fit(X_rate, y_rate)

    joblib.dump(attempts_pipe, attempts_pipe_path)
    joblib.dump(rate_pipe, rate_pipe_path)

    print(f"Saved: {attempts_pipe_path}")
    print(f"Saved: {rate_pipe_path}")


# -----------------------------
# 4) Predict a whole game (away, home)
# -----------------------------
def predict_game_fg3(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    features_path: str = PATH_TO_MODEL_dir + "fg3_features.joblib",
    attempts_pipe_path: str = PATH_TO_MODEL_dir + "fg3a_pipe.joblib",
    rate_pipe_path: str = PATH_TO_MODEL_dir + "fg3_rate_pipe.joblib",
    min_games_required: int = 10,
    recent_n: int = 5,
):
    FEATURES = joblib.load(features_path)
    attempts_pipe = joblib.load(attempts_pipe_path)
    rate_pipe = joblib.load(rate_pipe_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # latest team per player
    latest_team = (
        history.sort_values("date")
               .groupby("player")
               .tail(1)[["player", "team"]]
    )
    game_players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    # require enough games
    game_counts = history.groupby("player").size()
    game_players = [p for p in game_players if game_counts.get(p, 0) >= min_games_required]

    rows = []
    for player in game_players:
        ph = history[history["player"] == player].sort_values("date")
        prev = ph.iloc[-1]
        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        # Base expected line from trailing means
        base = recent_means(ph, n=recent_n)

        # Smarter expected fg3a with multipliers
        exp_fg3a = expected_fg3a_from_recent(history, ph, team=team, opp=opp, recent_n=recent_n)

        rows.append({
            "player": player,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": base["mp_minutes"],
            "fga": base["fga"],
            "fg3a": exp_fg3a,          # <- upgraded
            "pts": base["pts"],
            "usage": base["usage"],

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No players found for that matchup with sufficient history.")

    combined = pd.concat([history, today_df], ignore_index=True)

    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    X_today = combined.tail(len(today_df))[FEATURES].copy()

    # align indexes
    X_today = X_today.reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # require key rollings present to avoid garbage predictions
    min_required = ["fg3a_rolling_5", "fg3_pct_rolling_10", "min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1).to_numpy()

    X_ok = X_today.loc[mask]
    out = today_df.loc[mask, ["player", "team", "opp", "is_home", "fg3a"]].copy()

    # Predict attempts (optional improvement): model can refine our exp_fg3a
    # If you want to use model attempts instead of our exp_fg3a, uncomment:
    # pred_fg3a = np.clip(attempts_pipe.predict(X_ok), 0, None)
    # out["fg3a"] = pred_fg3a

    # Predict rate and convert to FG3
    pred_rate = np.clip(rate_pipe.predict(X_ok), 0, 1)
    out["pred_rate"] = pred_rate
    out["pred_fg3"] = out["fg3a"].to_numpy() * pred_rate

    # helpful debug columns (optional)
    out["exp_fg3a"] = out["fg3a"]
    out = out.drop(columns=["fg3a"])

    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out, X_today


# -----------------------------
# 5) Quick diagnostics
# -----------------------------
def quick_diagnostics(out: pd.DataFrame):
    print(out[["pred_fg3", "pred_rate"]].describe())
    print("\nTop 10:")
    print(out.head(10))


###  new 

In [129]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor


# ============================================================
# 1) FEATURE BUILDERS (NO LEAK)
# ============================================================

def build_features_no_leak(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates trailing/rolling features using shift(1) so each row uses ONLY past games.
    Requires columns:
    player, date, season, team, opp, mp_minutes, fga, fg3a, fg3, usage, is_home
    """
    df = df.sort_values(["player", "date"]).copy()
    g = df.groupby("player")

    # trailing rolling means (past-only)
    df["min_rolling_5"]  = g["mp_minutes"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)
    df["fga_rolling_5"]  = g["fga"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)
    df["fg3a_rolling_5"] = g["fg3a"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)
    df["fg3_rolling_5"]  = g["fg3"].shift(1).rolling(5).mean().reset_index(level=0, drop=True)

    made10 = g["fg3"].shift(1).rolling(10).sum().reset_index(level=0, drop=True)
    att10  = g["fg3a"].shift(1).rolling(10).sum().reset_index(level=0, drop=True)
    df["fg3_pct_rolling_10"] = made10 / att10

    # rest/context
    df["days_rest"] = g["date"].diff().dt.days
    df["back_to_back"] = (df["days_rest"] == 1).astype(int)

    # home flag
    df["home_game"] = df["is_home"].astype(int)

    # starter flag: if you don't have it, set constant 1
    if "starter_flag" not in df.columns:
        df["starter_flag"] = 1

    return df


def add_player_baselines(df: pd.DataFrame) -> pd.DataFrame:
    """
    Player identity WITHOUT using player name as a categorical feature.
    Past-only expanding stats via shift(1).
    """
    df = df.sort_values(["player", "date"]).copy()
    g = df.groupby("player")

    # "who is this player" baseline volume/skill/role
    df["player_fg3a_season_avg"] = g["fg3a"].shift(1).expanding().mean().reset_index(level=0, drop=True)

    made = g["fg3"].shift(1).expanding().sum()
    att  = g["fg3a"].shift(1).expanding().sum()
    df["player_fg3_pct_season"] = (made / att).reset_index(level=0, drop=True)

    df["player_min_season_avg"] = g["mp_minutes"].shift(1).expanding().mean().reset_index(level=0, drop=True)
    df["player_usage_season"]   = g["usage"].shift(1).expanding().mean().reset_index(level=0, drop=True)

    return df


# ============================================================
# 2) TRAIN TWO MODELS SEPARATELY
# ============================================================

FEATURES = [
    # form/opportunity
    "min_rolling_5", "fga_rolling_5", "fg3a_rolling_5",
    "fg3_rolling_5", "fg3_pct_rolling_10",
    # player identity baselines
    "player_fg3a_season_avg", "player_fg3_pct_season",
    "player_min_season_avg", "player_usage_season",
    # context
    "home_game", "days_rest", "back_to_back", "starter_flag",
]


joblib.dump(FEATURES, PATH_TO_MODEL_dir + "fg3_features_v2.joblib")



['../models/threes/fg3_features_v2.joblib']

In [130]:
def train_3fga(data):
    df = data.copy()
    train_a = df.dropna(subset=FEATURES + ["fg3a"]).copy()
    X_a = train_a[FEATURES]
    y_a = train_a["fg3a"]

    fg3a_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_depth=4,
            learning_rate=0.05,
            max_iter=400,
            random_state=42
        ))
    ])
    fg3a_pipe.fit(X_a, y_a)

    return fg3a_pipe


def train_3rate(data):
    df = data.copy()
    train_b = df.dropna(subset=FEATURES + ["fg3", "fg3a"]).copy()
    train_b = train_b[train_b["fg3a"] > 0]
    X_b = train_b[FEATURES]
    y_b = train_b["fg3"] / train_b["fg3a"]
    
    rate_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_depth=4,
            learning_rate=0.05,
            max_iter=400,
            random_state=42
        ))
    ])
    rate_pipe.fit(X_b, y_b)

    return rate_pipe


In [ ]:
def train_3fga(data):
    df = data.copy()
    train_a = df.dropna(subset=FEATURES + ["fg3a"]).copy()
    X_a = train_a[FEATURES]
    y_a = train_a["fg3a"]

    fg3a_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_depth=4,
            learning_rate=0.05,
            max_iter=400,
            random_state=42
        ))
    ])
    fg3a_pipe.fit(X_a, y_a)

    return fg3a_pipe


def train_3rate(data):
    df = data.copy()
    train_b = df.dropna(subset=FEATURES + ["fg3", "fg3a"]).copy()
    train_b = train_b[train_b["fg3a"] > 0]
    X_b = train_b[FEATURES]
    y_b = train_b["fg3"] / train_b["fg3a"]
    
    rate_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_depth=4,
            learning_rate=0.05,
            max_iter=400,
            random_state=42
        ))
    ])
    rate_pipe.fit(X_b, y_b)

    return rate_pipe

def train_models(
    csv_path="all_gamelogs_combined.csv",
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    rate_model_path=PATH_TO_MODEL_dir + "model_rate.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
):
    df = pd.read_csv(csv_path, parse_dates=["date"])

    # build features
    df = build_features_no_leak(df)
    df = add_player_baselines(df)

    # -------------------------
    # MODEL A: FG3A (attempts)
    # -------------------------

    fg3a_pipe = train_3fga(df)
    # -------------------------
    # MODEL B: RATE = FG3 / FG3A
    # -------------------------
    
    rate_pipe = train_3rate(df)
    
    # save everything
    joblib.dump(fg3a_pipe, fg3a_model_path)
    joblib.dump(rate_pipe, rate_model_path)
    joblib.dump(FEATURES, features_path)

    print("Saved:")
    print(" -", fg3a_model_path)
    print(" -", rate_model_path)
    print(" -", features_path)




In [1]:
# ============================================================
# 3) PREDICT A GAME USING BOTH MODELS
# ============================================================

def build_today_rows_for_game(history: pd.DataFrame, away_team: str, home_team: str, game_date,
                              min_games_required: int = 10, recent_n: int = 5) -> pd.DataFrame:
    """
    Creates one 'today row' per player in the matchup.
    Fills required raw inputs using recent averages (not fixed numbers).
    """
    history = history.sort_values(["player", "date"]).copy()

    latest_team = (
        history.sort_values("date")
               .groupby("player")
               .tail(1)[["player", "team", "season"]]
    )
    players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    game_counts = history.groupby("player").size()
    players = [p for p in players if game_counts.get(p, 0) >= min_games_required]

    rows = []
    for p in players:
        ph = history[history["player"] == p].sort_values("date")
        prev = ph.iloc[-1]
        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        # recent averages used as "expected line"
        mp = float(ph["mp_minutes"].tail(recent_n).mean())
        fga = float(ph["fga"].tail(recent_n).mean())
        fg3a = float(ph["fg3a"].tail(recent_n).mean())
        pts = float(ph["pts"].tail(recent_n).mean())
        usage = float(ph["usage"].tail(recent_n).mean())

        rows.append({
            "player": p,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": mp,
            "fga": fga,
            "fg3a": fg3a,
            "pts": pts,
            "usage": usage,

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,   # unknown target
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No eligible players found for this matchup.")
    return today_df


def predict_game_fg3_two_models(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    rate_model_path=PATH_TO_MODEL_dir + "model_rate.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
):
    fg3a_pipe = joblib.load(fg3a_model_path)
    rate_pipe = joblib.load(rate_model_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])

    today_df = build_today_rows_for_game(history, away_team, home_team, game_date)

    combined = pd.concat([history, today_df], ignore_index=True)

    # rebuild features on the combined frame
    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    # grab the feature rows corresponding to today rows
    X_today = combined.tail(len(today_df))[FEATURES].copy()
    X_today = X_today.reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # Optional: require key rollings exist
    min_required = ["fg3a_rolling_5", "fg3_pct_rolling_10", "min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1).to_numpy()

    X_ok = X_today.loc[mask]
    out = today_df.loc[mask, ["player", "team", "opp", "is_home"]].copy()

    # Predict both pieces
    pred_fg3a = np.clip(fg3a_pipe.predict(X_ok), 0, None)      # attempts
    pred_rate = np.clip(rate_pipe.predict(X_ok), 0, 1)         # accuracy

    out["pred_fg3a"] = pred_fg3a
    out["pred_rate"] = pred_rate
    out["pred_fg3"]  = pred_fg3a * pred_rate

    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out


NameError: name 'pd' is not defined

In [133]:
# ============================================================
# 4) HOW TO RUN
# ============================================================

train_models(csv_path=PATH_GAMLOGS_COMBINED)

# 2) Predict a game
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])

out = predict_game_fg3_two_models(
    history_df=history,
    away_team="HOU",
    home_team="DET",
    game_date="2026-01-23"
)

print(out.head(25))


Saved:
 - ../models/threes/model_fg3a.joblib
 - ../models/threes/model_rate.joblib
 - ../models/threes/features.joblib
                 player team  opp  is_home  pred_fg3a  pred_rate  pred_fg3
0          Caris LeVert  DET  HOU        1   3.295311   0.331072  1.090985
1            Tari Eason  HOU  DET        0   3.281299   0.331072  1.086346
2         Armoni Brooks  HOU  DET        0   3.233025   0.333557  1.078397
3             Paul Reed  DET  HOU        1   3.253857   0.325705  1.059796
4   Dorian Finney-Smith  HOU  DET        0   3.206413   0.329887  1.057754
5          Steven Adams  HOU  DET        0   3.238761   0.326463  1.057335
6            Jaden Ivey  DET  HOU        1   3.284132   0.321096  1.054520
7         Aaron Holiday  HOU  DET        0   3.321144   0.317126  1.053222
8      Jabari Smith Jr.  HOU  DET        0   3.232449   0.325705  1.052824
9         Marcus Sasser  DET  HOU        1   3.212259   0.325705  1.046248
10        Reed Sheppard  HOU  DET        0   3.254867   

In [134]:
def predict_game_fg3_two_models_upgraded(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    rate_model_path=PATH_TO_MODEL_dir + "model_rate.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
    min_games_required: int = 10,
    recent_n: int = 5,
):
    fg3a_pipe = joblib.load(fg3a_model_path)
    rate_pipe = joblib.load(rate_model_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # --- league baselines ---
    league_fga = history["fga"].mean()
    league_fg3a = history["fg3a"].mean()

    team_fga_avg = history.groupby("team")["fga"].mean()
    opp_fg3a_allowed = history.groupby("opp")["fg3a"].mean()

    # --- build today's rows ---
    today_rows = []
    latest_team = (
        history.sort_values("date")
        .groupby("player")
        .tail(1)[["player", "team", "season"]]
    )

    players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"]
    game_counts = history.groupby("player").size()
    players = [p for p in players if game_counts.get(p, 0) >= min_games_required]

    for p in players:
        ph = history[history["player"] == p]
        prev = ph.iloc[-1]

        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        # base expectations
        base_fg3a = ph["fg3a"].tail(recent_n).mean()
        base_usage = ph["usage"].tail(recent_n).mean()

        # multipliers
        pace_mult = ((team_fga_avg[team] + team_fga_avg.get(opp, league_fga)) / 2) / league_fga
        pace_mult = np.clip(pace_mult, 0.85, 1.15)

        opp_mult = opp_fg3a_allowed.get(opp, league_fg3a) / league_fg3a
        opp_mult = np.clip(opp_mult, 0.80, 1.25)

        usage_trend = (
            ph["usage"].tail(5).mean() / ph["usage"].tail(10).mean()
            if ph["usage"].tail(10).mean() > 0 else 1.0
        )
        usage_trend = np.clip(usage_trend, 0.85, 1.20)

        exp_fg3a = base_fg3a * pace_mult * opp_mult * usage_trend

        today_rows.append({
            "player": p,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,
            "mp_minutes": ph["mp_minutes"].tail(recent_n).mean(),
            "fga": ph["fga"].tail(recent_n).mean(),
            "fg3a": exp_fg3a,
            "pts": ph["pts"].tail(recent_n).mean(),
            "usage": base_usage,
            "is_home": is_home,
            "starter_flag": int(prev.get("starter_flag", 1)),
            "fg3": np.nan,
        })

    today_df = pd.DataFrame(today_rows)
    combined = pd.concat([history, today_df], ignore_index=True)

    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    X_today = combined.tail(len(today_df))[FEATURES].reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # predict
    pred_fg3a = np.clip(fg3a_pipe.predict(X_today), 0, None)
    pred_rate = np.clip(rate_pipe.predict(X_today), 0, 1)

    out = today_df[["player", "team", "opp", "is_home"]].copy()
    out["pred_fg3a"] = pred_fg3a
    out["pred_rate"] = pred_rate
    out["pred_fg3"] = pred_fg3a * pred_rate

    return out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)


In [135]:
out = predict_game_fg3_two_models_upgraded(history, "GSW", "DET", "2026-01-23")
print(out.head(15))


               player team  opp  is_home  pred_fg3a  pred_rate  pred_fg3
0           Saben Lee  DET  GSW        1   5.135422   0.334386  1.717215
1       Killian Hayes  DET  GSW        1   5.053035   0.334386  1.689666
2       Frank Jackson  DET  GSW        1   5.053035   0.331762  1.676406
3         Braxton Key  DET  GSW        1   5.051040   0.331762  1.675744
4     Jamorko Pickett  DET  GSW        1   5.051040   0.331762  1.675744
5       Isaiah Livers  DET  GSW        1   5.051040   0.331762  1.675744
6          Saddiq Bey  DET  GSW        1   4.978324   0.334386  1.664684
7      Ron Harper Jr.  DET  GSW        1   4.936306   0.331762  1.637680
8       Malik Beasley  DET  GSW        1   4.891120   0.331762  1.622689
9        Nico Mannion  GSW  DET        0   4.842924   0.331762  1.606699
10  Deividas Sirvydis  DET  GSW        1   4.772135   0.334386  1.595737
11         Joe Harris  DET  GSW        1   4.761229   0.334386  1.592090
12      Buddy Boeheim  DET  GSW        1   4.737105

In [136]:
def compute_final_rate(X_row, league_fg3_pct):
    """
    X_row = one row of X_today (features)
    """
    player_base = X_row["player_fg3_pct_season"]
    recent_form = X_row["fg3_pct_rolling_10"]

    # handle NaNs safely
    if np.isnan(recent_form):
        recent_form = player_base

    # weighted blend
    rate = (
        0.55 * player_base +
        0.35 * recent_form +
        0.10 * league_fg3_pct
    )

    return float(np.clip(rate, 0.15, 0.50))


In [137]:
import numpy as np
import pandas as pd
import joblib

def expected_fg3a_simple(ph: pd.DataFrame, recent_n: int = 5) -> float:
    return float(ph["fg3a"].tail(recent_n).mean())

def compute_final_rate(X_row: pd.Series, league_fg3_pct: float) -> float:
    """
    Deterministic rate model:
    - player baseline skill (season-to-date)
    - recent form (rolling 10)
    - small league anchor
    """
    player_base = X_row.get("player_fg3_pct_season", np.nan)
    recent_form = X_row.get("fg3_pct_rolling_10", np.nan)

    # fallbacks
    if np.isnan(player_base):
        player_base = league_fg3_pct
    if np.isnan(recent_form):
        recent_form = player_base

    # weighted blend (tuneable)
    rate = (
        0.55 * player_base +
        0.35 * recent_form +
        0.10 * league_fg3_pct
    )

    # clamp to sane NBA range
    return float(np.clip(rate, 0.15, 0.50))


def predict_game_fg3_two_models_better(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    fg3a_model_path,
    features_path,
    min_games_required: int = 10,
    recent_n: int = 5,
    fg3a_blend: float = 0.25,   # 0.25 = 25% model, 75% recent-avg
):
    # Load only the FG3A model + features
    fg3a_pipe = joblib.load(fg3a_model_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # latest team per player
    latest_team = (
        history.sort_values("date")
              .groupby("player")
              .tail(1)[["player", "team", "season"]]
    )

    players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    # require enough games
    game_counts = history.groupby("player").size()
    players = [p for p in players if game_counts.get(p, 0) >= min_games_required]

    rows = []
    for p in players:
        ph = history[history["player"] == p].sort_values("date")
        prev = ph.iloc[-1]

        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        rows.append({
            "player": p,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            # expected line from recent games
            "mp_minutes": float(ph["mp_minutes"].tail(recent_n).mean()),
            "fga": float(ph["fga"].tail(recent_n).mean()),
            "fg3a": expected_fg3a_simple(ph, recent_n=recent_n),
            "pts": float(ph["pts"].tail(recent_n).mean()),
            "usage": float(ph["usage"].tail(recent_n).mean()),

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No eligible players found for this matchup.")

    # Append + rebuild features
    combined = pd.concat([history, today_df], ignore_index=True)
    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    # Today feature rows
    X_today = combined.tail(len(today_df))[FEATURES].copy().reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # Optional filter: require key rollings exist
    min_required = ["fg3a_rolling_5", "fg3_pct_rolling_10", "min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1).to_numpy()

    X_ok = X_today.loc[mask].copy()
    out = today_df.loc[mask, ["player", "team", "opp", "is_home", "fg3a"]].copy()

    # -------------------------
    # 1) Attempts: blend expected fg3a + model fg3a
    # -------------------------
    model_fg3a = np.clip(fg3a_pipe.predict(X_ok), 0, None)
    expected_fg3a = out["fg3a"].to_numpy()

    final_fg3a = (1.0 - fg3a_blend) * expected_fg3a + fg3a_blend * model_fg3a
    final_fg3a = np.clip(final_fg3a, 0, None)

    # -------------------------
    # 2) Rate: computed from features (NO rate model)
    # -------------------------
    league_fg3_pct = history["fg3"].sum() / max(history["fg3a"].sum(), 1)

    final_rate = np.array([compute_final_rate(X_ok.iloc[i], league_fg3_pct) for i in range(len(X_ok))])

    # -------------------------
    # 3) Final FG3
    # -------------------------
    out["pred_fg3a"] = final_fg3a
    out["pred_rate"] = final_rate
    out["pred_fg3"] = final_fg3a * final_rate

    out = out.drop(columns=["fg3a"])
    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out


In [138]:
out = predict_game_fg3_two_models_better(
    history_df=history,
    away_team="OKCT",
    home_team="GSW",
    game_date="2026-01-23",
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
)
print(out.head(20))


                  player team   opp  is_home  pred_fg3a  pred_rate  pred_fg3
0          Stephen Curry  GSW  OKCT        1   8.256356   0.373752  3.085830
1            Moses Moody  GSW  OKCT        1   5.457743   0.374043  2.041433
2     Brandin Podziemski  GSW  OKCT        1   5.151592   0.355387  1.830807
3         Draymond Green  GSW  OKCT        1   4.562252   0.390948  1.783605
4      De'Anthony Melton  GSW  OKCT        1   4.699619   0.371860  1.747602
5            Buddy Hield  GSW  OKCT        1   4.470877   0.370025  1.654334
6             Al Horford  GSW  OKCT        1   4.406339   0.348527  1.535728
7           Quinten Post  GSW  OKCT        1   3.942554   0.357503  1.409473
8           Will Richard  GSW  OKCT        1   3.072107   0.376687  1.157223
9             Gui Santos  GSW  OKCT        1   2.907595   0.352546  1.025062
10          Jimmy Butler  GSW  OKCT        1   3.038743   0.315853  0.959797
11      Jonathan Kuminga  GSW  OKCT        1   2.300949   0.348528  0.801945

In [139]:
def debug_out(out):
    print("pred_fg3 max:", out["pred_fg3"].max())
    print("pred_fg3a max:", out["pred_fg3a"].max())
    print("pred_rate max:", out["pred_rate"].max())
    print("\nTop 10 by pred_fg3a:")
    print(out.sort_values("pred_fg3a", ascending=False).head(10)[["player","team","pred_fg3a","pred_rate","pred_fg3"]])
    print("\nTop 10 by pred_rate:")
    print(out.sort_values("pred_rate", ascending=False).head(10)[["player","team","pred_fg3a","pred_rate","pred_fg3"]])

debug_out(out)

pred_fg3 max: 3.085830329172085
pred_fg3a max: 8.256356018548754
pred_rate max: 0.39094843929532275

Top 10 by pred_fg3a:
                player team  pred_fg3a  pred_rate  pred_fg3
0        Stephen Curry  GSW   8.256356   0.373752  3.085830
1          Moses Moody  GSW   5.457743   0.374043  2.041433
2   Brandin Podziemski  GSW   5.151592   0.355387  1.830807
4    De'Anthony Melton  GSW   4.699619   0.371860  1.747602
3       Draymond Green  GSW   4.562252   0.390948  1.783605
5          Buddy Hield  GSW   4.470877   0.370025  1.654334
6           Al Horford  GSW   4.406339   0.348527  1.535728
7         Quinten Post  GSW   3.942554   0.357503  1.409473
8         Will Richard  GSW   3.072107   0.376687  1.157223
10        Jimmy Butler  GSW   3.038743   0.315853  0.959797

Top 10 by pred_rate:
                player team  pred_fg3a  pred_rate  pred_fg3
3       Draymond Green  GSW   4.562252   0.390948  1.783605
8         Will Richard  GSW   3.072107   0.376687  1.157223
1          Moses

#### lAtest

In [ ]:
import numpy as np
import pandas as pd
import joblib
import numpy as np
import pandas as pd
import joblib


# ----------------------------
# 1) CEILING-AWARE EXPECTED FG3A
# ----------------------------
def expected_fg3a_ceiling(ph: pd.DataFrame, recent_n: int = 5) -> float:
    """
    Better than mean(last N):
    - base = trailing mean (stable)
    - ceiling = recent 80th percentile (captures spike behavior)
    - blend them so stars can pop
    - cap at recent 95th percentile
    """
    tail = ph["fg3a"].tail(max(10, recent_n * 2)).dropna()
    if tail.empty:
        return 0.0

    base = float(tail.tail(recent_n).mean())
    ceiling = float(tail.quantile(0.80))
    exp = 0.65 * base + 0.35 * ceiling

    cap = float(tail.quantile(0.95))
    return float(np.clip(exp, 0, cap))


# ----------------------------
# 2) COMPUTED RATE (NO RATE MODEL) + ELITE BOOST
# ----------------------------
def compute_final_rate(X_row: pd.Series, league_fg3_pct: float) -> float:
    """
    Stabilized rate:
    - Use bayesian-shrunk baseline so tiny samples don't dominate
    - Blend baseline + recent form + league
    """

    # Inputs from features
    player_pct = X_row.get("player_fg3_pct_season", np.nan)
    recent_form = X_row.get("fg3_pct_rolling_10", np.nan)

    # Bayesian shrinkage prior
    prior_pct = league_fg3_pct
    prior_att = 80.0  # bigger = more conservative early season

    # If player_pct missing, start at league
    if np.isnan(player_pct):
        player_pct = prior_pct

    # Stabilize recent_form too
    if np.isnan(recent_form):
        recent_form = player_pct

    # Clamp both to sane shooting range before blending
    player_pct = float(np.clip(player_pct, 0.20, 0.50))
    recent_form = float(np.clip(recent_form, 0.15, 0.60))

    # Blend (baseline dominates; recent is a smaller modifier)
    rate = (
        0.70 * player_pct +
        0.20 * recent_form +
        0.10 * prior_pct
    )

    # Optional elite bump ONLY if baseline is truly elite
    if player_pct >= 0.40:
        rate *= 1.03

    return float(np.clip(rate, 0.18, 0.45))


# ----------------------------
# 3) PREDICT A GAME: FG3A MODEL + COMPUTED RATE
# ----------------------------
def predict_game_fg3_cnp(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    fg3a_model_path,
    features_path,
    min_games_required: int = 10,
    recent_n: int = 5,
    fg3a_blend: float = 0.25,   # 25% model FG3A, 75% expected FG3A
):
    # Load FG3A model + features
    fg3a_pipe = joblib.load(fg3a_model_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # latest team per player
    latest_team = (
        history.sort_values("date")
              .groupby("player")
              .tail(1)[["player", "team", "season"]]
    )
    players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    # require enough games
    game_counts = history.groupby("player").size()
    players = [p for p in players if game_counts.get(p, 0) >= min_games_required]

    # build today rows using recent averages (no fixed numbers)
    rows = []
    for p in players:
        ph = history[history["player"] == p].sort_values("date")
        prev = ph.iloc[-1]

        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        rows.append({
            "player": p,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": float(ph["mp_minutes"].tail(recent_n).mean()),
            "fga": float(ph["fga"].tail(recent_n).mean()),
            # upgraded attempts expectation
            "fg3a": expected_fg3a_ceiling(ph, recent_n=recent_n),
            "pts": float(ph["pts"].tail(recent_n).mean()),
            "usage": float(ph["usage"].tail(recent_n).mean()),

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No eligible players found for this matchup.")

    # Append + rebuild features
    combined = pd.concat([history, today_df], ignore_index=True)

    # NOTE: These must exist in your environment
    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    # Today feature rows
    X_today = combined.tail(len(today_df))[FEATURES].copy().reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # Loosen mask: only require minutes rolling exists
    min_required = ["min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1).to_numpy()

    X_ok = X_today.loc[mask].copy()
    out = today_df.loc[mask, ["player", "team", "opp", "is_home", "fg3a"]].copy()

    # -------------------------
    # Attempts: blend expected FG3A + model FG3A
    # -------------------------
    model_fg3a = np.clip(fg3a_pipe.predict(X_ok), 0, None)
    expected_fg3a = out["fg3a"].to_numpy()

    final_fg3a = (1.0 - fg3a_blend) * expected_fg3a + fg3a_blend * model_fg3a
    final_fg3a = np.clip(final_fg3a, 0, None)

    # -------------------------
    # Rate: computed, no rate model
    # -------------------------
    league_fg3_pct = history["fg3"].sum() / max(history["fg3a"].sum(), 1)

    final_rate = np.array([compute_final_rate(X_ok.iloc[i], league_fg3_pct) for i in range(len(X_ok))])

    # -------------------------
    # Final FG3
    # -------------------------
    out["pred_fg3a"] = final_fg3a
    out["pred_rate"] = final_rate
    out["pred_fg3"] = out["pred_fg3a"] * out["pred_rate"]

    out = out.drop(columns=["fg3a"])
    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out


#----------------------------
#EXAMPLE USAGE
#----------------------------
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])
out = predict_game_fg3_cnp(
    history_df=history,
    away_team="DET",
    home_team="GSW",
    game_date="2026-01-23",
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
)
print(out.head(25))


# ----------------------------
# 1) CEILING-AWARE EXPECTED FG3A
# ----------------------------
def expected_fg3a_ceiling(ph: pd.DataFrame, recent_n: int = 5) -> float:
    """
    Better than mean(last N):
    - base = trailing mean (stable)
    - ceiling = recent 80th percentile (captures spike behavior)
    - blend them so stars can pop
    - cap at recent 95th percentile
    """
    tail = ph["fg3a"].tail(max(10, recent_n * 2)).dropna()
    if tail.empty:
        return 0.0

    base = float(tail.tail(recent_n).mean())
    ceiling = float(tail.quantile(0.80))
    exp = 0.65 * base + 0.35 * ceiling

    cap = float(tail.quantile(0.95))
    return float(np.clip(exp, 0, cap))


# ----------------------------
# 2) COMPUTED RATE (NO RATE MODEL) + ELITE BOOST
# ----------------------------
def compute_final_rate(X_row: pd.Series, league_fg3_pct: float) -> float:
    """
    Stabilized rate:
    - Use bayesian-shrunk baseline so tiny samples don't dominate
    - Blend baseline + recent form + league
    """

    # Inputs from features
    player_pct = X_row.get("player_fg3_pct_season", np.nan)
    recent_form = X_row.get("fg3_pct_rolling_10", np.nan)

    # Bayesian shrinkage prior
    prior_pct = league_fg3_pct
    prior_att = 80.0  # bigger = more conservative early season

    # If player_pct missing, start at league
    if np.isnan(player_pct):
        player_pct = prior_pct

    # Stabilize recent_form too
    if np.isnan(recent_form):
        recent_form = player_pct

    # Clamp both to sane shooting range before blending
    player_pct = float(np.clip(player_pct, 0.20, 0.50))
    recent_form = float(np.clip(recent_form, 0.15, 0.60))

    # Blend (baseline dominates; recent is a smaller modifier)
    rate = (
        0.70 * player_pct +
        0.20 * recent_form +
        0.10 * prior_pct
    )

    # Optional elite bump ONLY if baseline is truly elite
    if player_pct >= 0.40:
        rate *= 1.03

    return float(np.clip(rate, 0.18, 0.45))


# ----------------------------
# 3) PREDICT A GAME: FG3A MODEL + COMPUTED RATE
# ----------------------------
def predict_game_fg3(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    fg3a_model_path,
    features_path,
    min_games_required: int = 10,
    recent_n: int = 5,
    fg3a_blend: float = 0.25,   # 25% model FG3A, 75% expected FG3A
):
    # Load FG3A model + features
    fg3a_pipe = joblib.load(fg3a_model_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # latest team per player
    latest_team = (
        history.sort_values("date")
              .groupby("player")
              .tail(1)[["player", "team", "season"]]
    )
    players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    # require enough games
    game_counts = history.groupby("player").size()
    players = [p for p in players if game_counts.get(p, 0) >= min_games_required]

    # build today rows using recent averages (no fixed numbers)
    rows = []
    for p in players:
        ph = history[history["player"] == p].sort_values("date")
        prev = ph.iloc[-1]

        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        rows.append({
            "player": p,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": float(ph["mp_minutes"].tail(recent_n).mean()),
            "fga": float(ph["fga"].tail(recent_n).mean()),
            # upgraded attempts expectation
            "fg3a": expected_fg3a_ceiling(ph, recent_n=recent_n),
            "pts": float(ph["pts"].tail(recent_n).mean()),
            "usage": float(ph["usage"].tail(recent_n).mean()),

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No eligible players found for this matchup.")

    # Append + rebuild features
    combined = pd.concat([history, today_df], ignore_index=True)

    # NOTE: These must exist in your environment
    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    # Today feature rows
    X_today = combined.tail(len(today_df))[FEATURES].copy().reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # Loosen mask: only require minutes rolling exists
    min_required = ["min_rolling_5"]
    mask = X_today[min_required].notna().all(axis=1).to_numpy()

    X_ok = X_today.loc[mask].copy()
    out = today_df.loc[mask, ["player", "team", "opp", "is_home", "fg3a"]].copy()

    # -------------------------
    # Attempts: blend expected FG3A + model FG3A
    # -------------------------
    model_fg3a = np.clip(fg3a_pipe.predict(X_ok), 0, None)
    expected_fg3a = out["fg3a"].to_numpy()

    final_fg3a = (1.0 - fg3a_blend) * expected_fg3a + fg3a_blend * model_fg3a
    final_fg3a = np.clip(final_fg3a, 0, None)

    # -------------------------
    # Rate: computed, no rate model
    # -------------------------
    league_fg3_pct = history["fg3"].sum() / max(history["fg3a"].sum(), 1)

    final_rate = np.array([compute_final_rate(X_ok.iloc[i], league_fg3_pct) for i in range(len(X_ok))])

    # -------------------------
    # Final FG3
    # -------------------------
    out["pred_fg3a"] = final_fg3a
    out["pred_rate"] = final_rate
    out["pred_fg3"] = out["pred_fg3a"] * out["pred_rate"]

    out = out.drop(columns=["fg3a"])
    out = out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)
    return out


#----------------------------
#EXAMPLE USAGE
#----------------------------
history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])
out = predict_game_fg3(
    history_df=history,
    away_team="DET",
    home_team="GSW",
    game_date="2026-01-23",
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
)
print(out.head(25))


                player team  opp  is_home  pred_fg3a  pred_rate  pred_fg3
0        Malik Beasley  DET  GSW        0   9.952780   0.318732  3.172265
1           Saddiq Bey  DET  GSW        0   8.557081   0.335404  2.870076
2      Duncan Robinson  DET  GSW        0   7.254684   0.340370  2.469280
3          Moses Moody  GSW  DET        1   5.571657   0.356740  1.987633
4      Jamorko Pickett  DET  GSW        0   5.305260   0.335402  1.779394
5          Buddy Hield  GSW  DET        1   5.059043   0.344473  1.742702
6        Killian Hayes  DET  GSW        0   4.870759   0.345404  1.682378
7           Al Horford  GSW  DET        1   4.613819   0.356738  1.645924
8      Cade Cunningham  DET  GSW        0   4.880807   0.330825  1.614695
9        Frank Jackson  DET  GSW        0   4.645759   0.335404  1.558204
10          Jaden Ivey  DET  GSW        0   4.562867   0.338712  1.545496
11       Marcus Sasser  DET  GSW        0   4.384088   0.348552  1.528084
12       Isaiah Livers  DET  GSW      

In [147]:
import numpy as np
import pandas as pd
import joblib


# ============================================================
# 1) CEILING-AWARE EXPECTED FG3A
# ============================================================
def expected_fg3a_ceiling(ph: pd.DataFrame, recent_n: int = 5) -> float:
    """
    - base = trailing mean
    - ceiling = recent 80th percentile (captures spike behavior)
    - cap at recent 95th percentile
    """
    tail = ph["fg3a"].tail(max(10, recent_n * 2)).dropna()
    if tail.empty:
        return 0.0

    base = float(tail.tail(recent_n).mean())
    ceiling = float(tail.quantile(0.80))
    exp = 0.65 * base + 0.35 * ceiling

    cap = float(tail.quantile(0.95))
    return float(np.clip(exp, 0, cap))


# ============================================================
# 2) TRUE BAYESIAN-SHRUNK RATE (NO RATE MODEL)
# ============================================================
def compute_final_rate_bayes(X_row: pd.Series, league_fg3_pct: float) -> float:
    """
    Uses true bayesian shrinkage:
      base = (made + prior_made) / (att + prior_att)
    Then blends in recent form in a volume-aware way.
    Requires these features exist in X_row:
      - player_fg3_made_sum
      - player_fg3_att_sum
      - fg3_pct_rolling_10
    """
    made = X_row.get("player_fg3_made_sum", np.nan)
    att = X_row.get("player_fg3_att_sum", np.nan)
    recent_form = X_row.get("fg3_pct_rolling_10", np.nan)

    # prior
    prior_att = 80.0
    prior_made = prior_att * league_fg3_pct

    # shrunken baseline
    if np.isnan(att) or att <= 0 or np.isnan(made):
        base = league_fg3_pct
        att_val = 0.0
    else:
        base = (made + prior_made) / (att + prior_att)
        att_val = float(att)

    # recent fallback
    if np.isnan(recent_form):
        recent_form = base

    # clamp inputs
    base = float(np.clip(base, 0.20, 0.50))
    recent_form = float(np.clip(recent_form, 0.15, 0.60))

    # volume-aware blending: more attempts -> trust recent more
    vol_weight = float(np.clip(att_val / 200.0, 0.0, 1.0))  # att=200 => full
    w_recent = 0.10 + 0.20 * vol_weight                     # 0.10..0.30
    w_base = 0.85 - 0.20 * vol_weight                       # 0.85..0.65
    w_league = 1.0 - (w_base + w_recent)                    # remaining

    rate = (w_base * base) + (w_recent * recent_form) + (w_league * league_fg3_pct)

    # small elite bump if truly elite baseline
    if base >= 0.40:
        rate *= 1.02

    return float(np.clip(rate, 0.18, 0.45))


# ============================================================
# 3) PREDICT A GAME: FG3A MODEL + BAYESIAN RATE
# ============================================================
def predict_game_fg3(
    history_df: pd.DataFrame,
    away_team: str,
    home_team: str,
    game_date,
    fg3a_model_path,
    features_path,
    min_games_required: int = 10,
    recent_n: int = 5,
    fg3a_blend: float = 0.25,   # 25% model FG3A, 75% expected FG3A
):
    fg3a_pipe = joblib.load(fg3a_model_path)
    FEATURES = joblib.load(features_path)

    history = history_df.copy()
    history["date"] = pd.to_datetime(history["date"])
    history = history.sort_values(["player", "date"])

    # players on either team (based on latest team)
    latest_team = (
        history.sort_values("date")
              .groupby("player")
              .tail(1)[["player", "team", "season"]]
    )
    players = latest_team[latest_team["team"].isin([away_team, home_team])]["player"].tolist()

    # require enough games
    game_counts = history.groupby("player").size()
    players = [p for p in players if game_counts.get(p, 0) >= min_games_required]

    rows = []
    for p in players:
        ph = history[history["player"] == p].sort_values("date")
        prev = ph.iloc[-1]

        team = prev["team"]
        is_home = 1 if team == home_team else 0
        opp = away_team if is_home else home_team

        rows.append({
            "player": p,
            "season": prev["season"],
            "date": pd.Timestamp(game_date),
            "team": team,
            "opp": opp,

            "mp_minutes": float(ph["mp_minutes"].tail(recent_n).mean()),
            "fga": float(ph["fga"].tail(recent_n).mean()),
            "fg3a": expected_fg3a_ceiling(ph, recent_n=recent_n),
            "pts": float(ph["pts"].tail(recent_n).mean()),
            "usage": float(ph["usage"].tail(recent_n).mean()),

            "is_home": int(is_home),
            "starter_flag": int(prev.get("starter_flag", 1)),

            "fg3": np.nan,
        })

    today_df = pd.DataFrame(rows)
    if today_df.empty:
        raise ValueError("No eligible players found for this matchup.")

    combined = pd.concat([history, today_df], ignore_index=True)

    # Must exist in your environment
    combined = build_features_no_leak(combined)
    combined = add_player_baselines(combined)

    X_today = combined.tail(len(today_df))[FEATURES].copy().reset_index(drop=True)
    today_df = today_df.reset_index(drop=True)

    # require only minutes rolling (avoid over-filtering)
    mask = X_today[["min_rolling_5"]].notna().all(axis=1).to_numpy()

    X_ok = X_today.loc[mask].copy()
    out = today_df.loc[mask, ["player", "team", "opp", "is_home", "fg3a"]].copy()

    # --- FG3A: blend expected + model ---
    model_fg3a = np.clip(fg3a_pipe.predict(X_ok), 0, None)
    expected_fg3a = out["fg3a"].to_numpy()
    final_fg3a = (1.0 - fg3a_blend) * expected_fg3a + fg3a_blend * model_fg3a
    final_fg3a = np.clip(final_fg3a, 0, None)

    # --- Rate: Bayesian computed ---
    league_fg3_pct = history["fg3"].sum() / max(history["fg3a"].sum(), 1)
    final_rate = np.array([compute_final_rate_bayes(X_ok.iloc[i], league_fg3_pct) for i in range(len(X_ok))])

    out["pred_fg3a"] = final_fg3a
    out["pred_rate"] = final_rate
    out["pred_fg3"] = out["pred_fg3a"] * out["pred_rate"]

    out = out.drop(columns=["fg3a"])
    return out.sort_values("pred_fg3", ascending=False).reset_index(drop=True)


# ============================================================
# EXAMPLE USAGE
# ============================================================



In [163]:
matchup = ("brk".upper(), "lac".upper(), "2026-01-25")

history = pd.read_csv(PATH_GAMLOGS_COMBINED, parse_dates=["date"])
out = predict_game_fg3(
    history_df=history,
    away_team=matchup[0],
    home_team=matchup[1],
    game_date=matchup[2],
    fg3a_model_path=PATH_TO_MODEL_dir + "model_fg3a.joblib",
    features_path=PATH_TO_MODEL_dir + "features.joblib",
)

out.sort_values("player")


,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3
24,Ben Simmons,LAC,BRK,1,1.088818,0.348826,0.379807
4,Bogdan Bogdanović,LAC,BRK,1,4.340131,0.352968,1.531929
5,Bradley Beal,LAC,BRK,1,4.131895,0.350455,1.448044
14,Brandon Boston Jr.,LAC,BRK,1,2.388876,0.344635,0.823290
3,Brook Lopez,LAC,BRK,1,4.552604,0.339576,1.545953
19,Cam Christie,LAC,BRK,1,1.883318,0.349773,0.658733
17,Chris Paul,LAC,BRK,1,2.101846,0.355308,0.746803
20,Daniel Oturu,LAC,BRK,1,1.449746,0.345826,0.501359
10,Derrick Jones Jr.,LAC,BRK,1,2.987303,0.354159,1.057980
26,Ivica Zubac,LAC,BRK,1,1.033181,0.349086,0.360669


In [165]:
data[data["player"] == "Michael Porter Jr."]

,player,season,date,team,opp,mp,fg,fga,fg3,fg3a,...,fg3_rolling_5,fg3_pct_rolling_10,days_rest,back_to_back,home_game,starter_flag,player_fg3a_season_avg,player_fg3_pct_season,player_min_season_avg,player_usage_season
18680,Michael Porter Jr.,2025,2024-10-24,DEN,OKC,32:10,5.0,17.0,3.0,10.0,...,0.8,0.363636,NaN,0,0,1,3.189239,0.356243,21.824792,0.454683
18681,Michael Porter Jr.,2025,2024-10-26,DEN,LAC,37:58,4.0,13.0,0.0,6.0,...,1.2,0.400000,2.0,0,0,1,3.189392,0.356240,21.825810,0.454704
18682,Michael Porter Jr.,2025,2024-10-28,DEN,TOR,39:22,6.0,12.0,1.0,4.0,...,0.8,0.375000,2.0,0,1,1,3.189273,0.356234,21.826246,0.454713
18683,Michael Porter Jr.,2025,2024-10-29,DEN,BRK,40:42,6.0,11.0,4.0,7.0,...,0.8,0.375000,1.0,1,1,1,3.189154,0.356228,21.826524,0.454719
18684,Michael Porter Jr.,2025,2024-11-01,DEN,MIN,38:53,11.0,18.0,3.0,7.0,...,0.6,0.333333,3.0,0,1,1,3.189089,0.356216,21.826792,0.454725
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18752,Michael Porter Jr.,2025,2025-04-04,DEN,GSW,40:38,9.0,15.0,3.0,7.0,...,0.2,0.166667,7.0,0,1,1,3.184442,0.356121,21.826043,0.454709
18753,Michael Porter Jr.,2025,2025-04-06,DEN,IND,38:46,6.0,14.0,1.0,9.0,...,0.2,0.166667,2.0,0,0,1,3.184323,0.356115,21.825090,0.454689
18754,Michael Porter Jr.,2025,2025-04-09,DEN,SAC,35:11,7.0,14.0,4.0,8.0,...,0.2,0.214286,3.0,0,1,1,3.184259,0.356120,21.824048,0.454668
18755,Michael Porter Jr.,2025,2025-04-11,DEN,MEM,29:31,2.0,8.0,0.0,2.0,...,0.2,0.187500,2.0,0,0,1,3.184195,0.356107,21.823027,0.454646


In [149]:
ph = history[history.player=="Stephen Curry"].sort_values("date")
print(ph["fg3a"].tail(15).to_list())
print("mean5:", ph["fg3a"].tail(5).mean())
print("p80:", ph["fg3a"].tail(15).quantile(0.80))
print("p95:", ph["fg3a"].tail(15).quantile(0.95))


[10.0, 11.0, 12.0, 10.0, 12.0, 15.0, 9.0, 12.0, 11.0, 8.0, 9.0, 8.0, 10.0, 7.0, 15.0]
mean5: 9.8
p80: 12.0
p95: 15.0


In [150]:
debug_out(out)

pred_fg3 max: 3.4503977501583933
pred_fg3a max: 9.977279266699707
pred_rate max: 0.36749218261413386

Top 10 by pred_fg3a:
              player team  pred_fg3a  pred_rate  pred_fg3
0      Malik Beasley  DET   9.977279   0.345826  3.450398
1         Saddiq Bey  DET   8.580048   0.354159  3.038700
2  Russell Westbrook  SAC   7.038120   0.352404  2.480265
3        Zach LaVine  SAC   5.977716   0.345826  2.067247
4    Jamorko Pickett  DET   5.311053   0.357189  1.897051
5      Killian Hayes  DET   4.735120   0.356540  1.688259
9      Frank Jackson  DET   4.643705   0.343048  1.593012
8      Tobias Harris  DET   4.579704   0.350371  1.604595
6         Jaden Ivey  DET   4.534691   0.362002  1.641567
7      Isaiah Livers  DET   4.455848   0.366280  1.632088

Top 10 by pred_rate:
               player team  pred_fg3a  pred_rate  pred_fg3
17       Caris LeVert  DET   3.495786   0.367492  1.284674
22         Keon Ellis  SAC   3.097220   0.366280  1.134450
7       Isaiah Livers  DET   4.455848   

# old

In [145]:
df = pd.read_csv("../../data/all_gamelogs_combined.csv", parse_dates=["date"])
df = build_features_no_leak(df)

FEATURES = [
    "min_rolling_5", "fga_rolling_5", "fg3a_rolling_5",
    "fg3_rolling_5", "fg3_pct_rolling_10",
    "days_rest", "back_to_back", "home_game", "starter_flag"
]

train_df = df.dropna(subset=FEATURES + ["fg3"])
X = train_df[FEATURES]
y = train_df["fg3"]


FileNotFoundError: [Errno 2] No such file or directory: '../../data/all_gamelogs_combined.csv'

In [ ]:
df.columns.to_list()

In [ ]:
data.info()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib

FEATURES = [
    "fg3a_per_min", "fg3a_share", "usage_proxy",
    "fg3_rolling_3", "fg3_rolling_5", "fg3a_rolling_5",
    "fg3_pct_rolling_10", "fg3_pct_season", "fg3_over_expected",
    "fg3_std_rolling_10", "minutes_rolling_5", "minutes_std_rolling_10",
    "pts_per_fga", "home_game", "starter_flag",
    "days_rest", "back_to_back",
    "opp_fg3_allowed_avg", "opp_pace_proxy", "team_fg3_rate"
]

df = pd.read_csv("../../data/all_gamelogs_combined.csv", parse_dates=["date"])
df = build_features(df)
df = df.dropna(subset=FEATURES + ["fg3"])

X = df[FEATURES]
y = df["fg3"]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

pipe.fit(X, y)

joblib.dump(pipe, "fg3_model.joblib")
joblib.dump(FEATURES, "fg3_features.joblib")


In [ ]:
today = pd.DataFrame([{
    "player": "Duncan Robinson",
    "date": pd.Timestamp("2026-01-23"),
    "season": "2025-26",
    "team": "DET",
    "opp": "HOU",
    "min": 34,
    "fga": 20,
    "fg3a": 11,
    "fg3": np.nan,  # UNKNOWN
    "pts": 28,
    "is_home": 1,
    "starter": 0
}])


history = pd.read_csv("../../data/all_gamelogs_combined.csv", parse_dates=["date"])

combined = pd.concat([history, today], ignore_index=True)
combined = build_features(combined)


In [ ]:
FEATURES = joblib.load("fg3_features.joblib")
pipe = joblib.load("fg3_model.joblib")

today_features = combined.iloc[-1:][FEATURES]

fg3_prediction = pipe.predict(today_features)[0]

print(f"Predicted FG3: {fg3_prediction:.2f}")


### three defense

In [19]:
import numpy as np
def build_3p_defense_rank_from_player_logs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build season-level 3P defensive rankings using player game logs.
    Defensive team is `opp`.
    """

    data = df.copy()

    # --- Aggregate opponent shooting ---
    season_def = (
        data.groupby(["season", "opp"])
        .agg(
            games_played=("date", "nunique"),
            opp_fg3a=("fg3a", "sum"),
            opp_fg3m=("fg3", "sum"),
        )
        .reset_index()
        .rename(columns={"opp": "team"})
    )

    # --- Per-game normalization ---
    season_def["opp_fg3a_pg"] = season_def["opp_fg3a"] / season_def["games_played"]
    season_def["opp_fg3m_pg"] = season_def["opp_fg3m"] / season_def["games_played"]

    season_def["opp_fg3_pct"] = (
        season_def["opp_fg3m"] / season_def["opp_fg3a"]
    ).replace([np.inf, np.nan], 0)

    # --- League ranks (lower = better defense) ---
    season_def["rank_fg3m_pg"] = (
        season_def.groupby("season")["opp_fg3m_pg"]
        .rank(method="min", ascending=True)
    )

    season_def["rank_fg3a_pg"] = (
        season_def.groupby("season")["opp_fg3a_pg"]
        .rank(method="min", ascending=True)
    )

    season_def["rank_fg3_pct"] = (
        season_def.groupby("season")["opp_fg3_pct"]
        .rank(method="min", ascending=True)
    )

    # --- Composite defensive score ---
    season_def["def_3p_score"] = (
        0.5 * season_def["rank_fg3m_pg"]
        + 0.3 * season_def["rank_fg3a_pg"]
        + 0.2 * season_def["rank_fg3_pct"]
    )

    # --- Final defensive position ---
    season_def["def_3p_rank"] = (
        season_def.groupby("season")["def_3p_score"]
        .rank(method="min", ascending=True)
        .astype(int)
    )

    return season_def[[
        "season",
        "team",
        "def_3p_rank",
        "opp_fg3a_pg",
        "opp_fg3m_pg",
        "opp_fg3_pct",
        "games_played"
    ]].sort_values(["season", "def_3p_rank"])


In [20]:
#from model_training.config import PATH_GAMLOGS_COMBINED
data = pd.read_csv("../data/all_gamelogs_combined.csv", parse_dates=["date"])

In [24]:
def_df  = build_3p_defense_rank_from_player_logs(data)
def_df[def_df["season"] == 2026].head(20)

,season,team,def_3p_rank,opp_fg3a_pg,opp_fg3m_pg,opp_fg3_pct,games_played
161,2026,IND,1,32.790698,11.209302,0.341844,43
171,2026,ORL,2,32.200000,11.675000,0.362578,40
173,2026,PHO,3,35.255814,12.255814,0.347625,43
156,2026,DAL,4,35.833333,11.976190,0.334219,42
160,2026,HOU,5,35.738095,12.214286,0.341772,42
158,2026,DET,6,35.441860,12.441860,0.351050,43
167,2026,MIN,7,34.977273,12.431818,0.355426,44
159,2026,GSW,8,35.386364,12.545455,0.354528,44
172,2026,PHI,9,35.476190,12.547619,0.353691,42
177,2026,TOR,10,36.404762,12.523810,0.344016,42


# pred


pred_fg3a >= 5.5 (you need attempts)

p_ge_2 >= 0.60 (solid probability)

optional: p_ge_3 >= 0.35 (avoids thin edges)

In [168]:
from datetime import datetime


_tdy = datetime.today().strftime("%Y-%m-%d")

out = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
out = out[out["pred_fg3a"] >= 5.5]
out = out[out['p_ge_2'] >= 0.6]
out = out[out['p_ge_3'] >= 0.35]

In [169]:
out[["player", "team", "opp", "p_ge_2"]].sort_values("p_ge_2", ascending=False).head(20)

,player,team,opp,p_ge_2
43,Jalen Brunson,NYK,DET,0.717410
80,Derrick White,BOS,GSW,0.698396
56,Anfernee Simons,CHI,TOR,0.689917
13,Tyrese Maxey,PHI,ATL,0.636552
57,Immanuel Quickley,TOR,CHI,0.629318
81,Moses Moody,GSW,BOS,0.628921
108,Jamal Murray,DEN,LAC,0.628846
95,Russell Westbrook,SAC,ORL,0.622982
6,Donovan Mitchell,CLE,BKN,0.621730


So for “beat baseline” analysis, first filter:

baseline_fg3 > 0.5 (or >1.0)

AND baseline should be player-specific (not default constant)

Then rank by:

p_over_baseline_2 (primary)

delta_fg3 (secondary)

pred_fg3a (tiebreaker)

In [170]:
out.columns

Index(['player', 'team', 'opp', 'is_home', 'pred_fg3a', 'pred_rate',
       'pred_fg3', 'baseline_fg3', 'delta_fg3', 'p_over_baseline_2', 'p_ge_2',
       'p_ge_3'],
      dtype='object')

In [171]:
out = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
out = out[out["baseline_fg3"] >= 0.5]
out.sort_values(['p_over_baseline_2', 'delta_fg3'], ascending=False).head(20)

out  = out[out['p_ge_3'] >= 0.45]
out = out[out["pred_fg3a"] >= 7.5]

out

,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,baseline_fg3,delta_fg3,p_over_baseline_2,p_ge_2,p_ge_3


In [172]:
out = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")
out.sort_values('baseline_fg3', ascending=False).head(20)

,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,baseline_fg3,delta_fg3,p_over_baseline_2,p_ge_2,p_ge_3
0,Tari Eason,HOU,CHA,0,5.175078,0.335094,1.734137,0.0,1.734137,0.517281,0.517281,0.251814
1,Reed Sheppard,HOU,CHA,0,5.258202,0.325278,1.710375,0.0,1.710375,0.509970,0.509970,0.245518
2,Kevin Durant,HOU,CHA,0,5.085769,0.332390,1.690460,0.0,1.690460,0.503786,0.503786,0.240261
3,Jabari Smith Jr.,HOU,CHA,0,4.522578,0.337840,1.527910,0.0,1.527910,0.451472,0.451472,0.198191
4,Alperen Şengün,HOU,CHA,0,1.060645,0.325856,0.345617,0.0,0.345617,0.047595,0.047595,0.005322
5,Amen Thompson,HOU,CHA,0,0.907578,0.316933,0.287642,0.0,0.287642,0.034230,0.034230,0.003202
6,Donovan Mitchell,CLE,BKN,1,6.460645,0.325856,2.105238,0.0,2.105238,0.621730,0.621730,0.351783
7,James Harden,CLE,BKN,1,5.760078,0.335094,1.930167,0.0,1.930167,0.574763,0.574763,0.304430
8,Sam Merrill,CLE,BKN,1,5.415078,0.316933,1.716219,0.0,1.716219,0.511775,0.511775,0.247064
9,Jaylon Tyson,CLE,BKN,1,3.220533,0.332574,1.071066,0.0,1.071066,0.290363,0.290363,0.093827


## ticks

In [173]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from model_training.threes.predict import predict_game_fg3


from datetime import datetime
_tdy = datetime.today().strftime("%Y-%m-%d")

In [174]:
# Example usage inside your workflow after you generate the per-game predictions dataframe `out`
# (the output of predict_game_fg3 for each matchup, concatenated)

from model_training.threes.selector import select_2plus_ticket, select_jackpot_ticket

out_all = pd.read_csv(f"../results/{_tdy}/all_matchups_fg3_predictions.csv")

ticket_2p = select_2plus_ticket(
    out_all,
    n_legs=10,
    min_pred_fg3a=5.8,
    min_p_ge_2=0.62,
    min_p_ge_3=0.30,
    max_per_team=3,
)

ticket_jackpot = select_jackpot_ticket(
    out_all,
    n_legs=3,
    min_pred_fg3a=5.5,
    min_p_over_baseline=0.18,
    min_delta_fg3=0.75,
    max_per_team=1,
    over_baseline_delta=2,
)

print("2+ Ticket:")
ticket_2 = ticket_2p[["player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]].copy()

print("\nJackpot Ticket (+2 over baseline):")
ticket_jack = ticket_jackpot[["player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","baseline_fg3","delta_fg3","p_over_baseline_2"]].copy()


2+ Ticket:

Jackpot Ticket (+2 over baseline):


In [175]:
import numpy as np
import pandas as pd


def select_matchup_coverage_ticket(
    df: pd.DataFrame,
    *,
    players_per_matchup: int = 2,
    insurance_per_matchup: int = 1,   # set 0 for "exactly 2"
    min_pred_fg3a: float = 5.6,
    min_p_ge_2: float = 0.60,
    min_p_ge_3: float = 0.30,
    max_legs: int | None = None,
) -> pd.DataFrame:
    """
    Matchup-coverage selector:
      - Groups rows into matchups (AWAY@HOME)
      - Picks top N players per matchup (plus optional insurance)
      - Ranks by: p_ge_2, then pred_fg3, then pred_fg3a

    Requires columns:
      player, team, opp, is_home, pred_fg3a, pred_fg3, p_ge_2, p_ge_3
    """
    req = {"player","team","opp","is_home","pred_fg3a","pred_fg3","p_ge_2","p_ge_3"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    out = df.copy()

    # Build matchup key as AWAY@HOME
    out["matchup_key"] = np.where(
        out["is_home"].astype(int) == 1,
        out["opp"].astype(str) + "@" + out["team"].astype(str),  # away@home
        out["team"].astype(str) + "@" + out["opp"].astype(str),
    )

    # Quality filters (tune these)
    pool = out[
        (out["pred_fg3a"] >= min_pred_fg3a)
        & (out["p_ge_2"] >= min_p_ge_2)
        & (out["p_ge_3"] >= min_p_ge_3)
    ].copy()

    if pool.empty:
        return pool

    # Rank within matchup
    pool = pool.sort_values(
        ["matchup_key","p_ge_2","pred_fg3","pred_fg3a"],
        ascending=[True, False, False, False],
    )
    pool["_rank"] = pool.groupby("matchup_key").cumcount()

    keep = players_per_matchup + insurance_per_matchup
    ticket = pool[pool["_rank"] < keep].copy()

    # Optional: cap total legs across slate
    if max_legs is not None and len(ticket) > max_legs:
        ticket = ticket.sort_values(["p_ge_2","pred_fg3"], ascending=False).head(max_legs)

    return (
        ticket
        .drop(columns=["_rank"])
        .sort_values(["matchup_key","p_ge_2","pred_fg3"], ascending=[True, False, False])
        .reset_index(drop=True)
    )

def assign_pencil_decision(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    conditions = [
        # Strong 3+ candidates
        (df["p_ge_3"] >= 0.35) & (df["pred_fg3a"] >= 6.0),

        # Solid 2+ candidates
        (df["p_ge_2"] >= 0.60) & (df["pred_fg3a"] >= 4.5),
    ]

    choices = [
        "3+",
        "2+",
    ]

    df["pencil"] = np.select(conditions, choices, default="coverage_only")

    return df


ticket_match = select_matchup_coverage_ticket(
    out_all,
    players_per_matchup=2,
    insurance_per_matchup=1,
    min_pred_fg3a=5.6,
    min_p_ge_2=0.60,
    min_p_ge_3=0.30,
)

print(ticket_match[["matchup_key","player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]]
      .to_string(index=False))



matchup_key                   player team opp  is_home  pred_fg3a  pred_rate  pred_fg3   p_ge_2   p_ge_3
    ATL@PHI             Tyrese Maxey  PHI ATL        1   6.457697   0.335094  2.163936 0.636552 0.367601
    ATL@PHI Nickeil Alexander-Walker  ATL PHI        0   6.585078   0.316933  2.087031 0.617039 0.346866
    BKN@CLE         Donovan Mitchell  CLE BKN        1   6.460645   0.325856  2.105238 0.621730 0.351783
    BOS@GSW            Derrick White  BOS GSW        0   7.672578   0.316933  2.431696 0.698396 0.438550
    BOS@GSW              Moses Moody  GSW BOS        1   6.315078   0.337840  2.133489 0.628921 0.359403
    DEN@LAC             Jamal Murray  DEN LAC        0   6.362367   0.335283  2.133191 0.628846 0.359323
    DET@NYK            Jalen Brunson  NYK DET        1   7.590769   0.332390  2.523097 0.717410 0.462098
    DET@NYK          Duncan Robinson  DET NYK        0   6.262697   0.335094  2.098592 0.620023 0.349989
    ORL@SAC        Russell Westbrook  SAC ORL        1 

In [176]:
pencel_tick = assign_pencil_decision(ticket_match)

In [177]:
pencel_tick[['player', 'team', 'pencil']]

,player,team,pencil
0,Tyrese Maxey,PHI,3+
1,Nickeil Alexander-Walker,ATL,2+
2,Donovan Mitchell,CLE,3+
3,Derrick White,BOS,3+
4,Moses Moody,GSW,3+
5,Jamal Murray,DEN,3+
6,Jalen Brunson,NYK,3+
7,Duncan Robinson,DET,2+
8,Russell Westbrook,SAC,3+
9,Anfernee Simons,CHI,3+


In [178]:
ticket_match_2 = select_matchup_coverage_ticket(
    out_all,
    players_per_matchup=2,
    insurance_per_matchup=0,
    min_pred_fg3a=5.6,
    min_p_ge_2=0.60,
    min_p_ge_3=0.30,
)


In [179]:

print(ticket_match_2[["matchup_key","player","team","opp","is_home","pred_fg3a","pred_rate","pred_fg3","p_ge_2","p_ge_3"]]
      .to_string(index=False))


matchup_key                   player team opp  is_home  pred_fg3a  pred_rate  pred_fg3   p_ge_2   p_ge_3
    ATL@PHI             Tyrese Maxey  PHI ATL        1   6.457697   0.335094  2.163936 0.636552 0.367601
    ATL@PHI Nickeil Alexander-Walker  ATL PHI        0   6.585078   0.316933  2.087031 0.617039 0.346866
    BKN@CLE         Donovan Mitchell  CLE BKN        1   6.460645   0.325856  2.105238 0.621730 0.351783
    BOS@GSW            Derrick White  BOS GSW        0   7.672578   0.316933  2.431696 0.698396 0.438550
    BOS@GSW              Moses Moody  GSW BOS        1   6.315078   0.337840  2.133489 0.628921 0.359403
    DEN@LAC             Jamal Murray  DEN LAC        0   6.362367   0.335283  2.133191 0.628846 0.359323
    DET@NYK            Jalen Brunson  NYK DET        1   7.590769   0.332390  2.523097 0.717410 0.462098
    DET@NYK          Duncan Robinson  DET NYK        0   6.262697   0.335094  2.098592 0.620023 0.349989
    ORL@SAC        Russell Westbrook  SAC ORL        1 

In [180]:
ticket_jack

,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,baseline_fg3,delta_fg3,p_over_baseline_2
0,Jalen Brunson,NYK,DET,1,7.590769,0.332390,2.523097,0.0,2.523097,0.717410
1,Derrick White,BOS,GSW,0,7.672578,0.316933,2.431696,0.0,2.431696,0.698396
2,Anfernee Simons,CHI,TOR,1,7.355179,0.325278,2.392475,0.0,2.392475,0.689917


In [166]:
ticket_match

,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,baseline_fg3,delta_fg3,p_over_baseline_2,p_ge_2,p_ge_3,matchup_key
0,Tyrese Maxey,PHI,ATL,1,6.457697,0.335094,2.163936,0.0,2.163936,0.636552,0.636552,0.367601,ATL@PHI
1,Nickeil Alexander-Walker,ATL,PHI,0,6.585078,0.316933,2.087031,0.0,2.087031,0.617039,0.617039,0.346866,ATL@PHI
2,Donovan Mitchell,CLE,BKN,1,6.460645,0.325856,2.105238,0.0,2.105238,0.621730,0.621730,0.351783,BKN@CLE
3,Derrick White,BOS,GSW,0,7.672578,0.316933,2.431696,0.0,2.431696,0.698396,0.698396,0.438550,BOS@GSW
4,Moses Moody,GSW,BOS,1,6.315078,0.337840,2.133489,0.0,2.133489,0.628921,0.628921,0.359403,BOS@GSW
5,Jamal Murray,DEN,LAC,0,6.362367,0.335283,2.133191,0.0,2.133191,0.628846,0.628846,0.359323,DEN@LAC
6,Jalen Brunson,NYK,DET,1,7.590769,0.332390,2.523097,0.0,2.523097,0.717410,0.717410,0.462098,DET@NYK
7,Duncan Robinson,DET,NYK,0,6.262697,0.335094,2.098592,0.0,2.098592,0.620023,0.620023,0.349989,DET@NYK
8,Russell Westbrook,SAC,ORL,1,6.475645,0.325856,2.110126,0.0,2.110126,0.622982,0.622982,0.353102,ORL@SAC
9,Anfernee Simons,CHI,TOR,1,7.355179,0.325278,2.392475,0.0,2.392475,0.689917,0.689917,0.428324,TOR@CHI


In [167]:
out_all.head(10)

,player,team,opp,is_home,pred_fg3a,pred_rate,pred_fg3,baseline_fg3,delta_fg3,p_over_baseline_2,p_ge_2,p_ge_3
0,Tari Eason,HOU,CHA,0,5.175078,0.335094,1.734137,0.0,1.734137,0.517281,0.517281,0.251814
1,Reed Sheppard,HOU,CHA,0,5.258202,0.325278,1.710375,0.0,1.710375,0.509970,0.509970,0.245518
2,Kevin Durant,HOU,CHA,0,5.085769,0.332390,1.690460,0.0,1.690460,0.503786,0.503786,0.240261
3,Jabari Smith Jr.,HOU,CHA,0,4.522578,0.337840,1.527910,0.0,1.527910,0.451472,0.451472,0.198191
4,Alperen Şengün,HOU,CHA,0,1.060645,0.325856,0.345617,0.0,0.345617,0.047595,0.047595,0.005322
5,Amen Thompson,HOU,CHA,0,0.907578,0.316933,0.287642,0.0,0.287642,0.034230,0.034230,0.003202
6,Donovan Mitchell,CLE,BKN,1,6.460645,0.325856,2.105238,0.0,2.105238,0.621730,0.621730,0.351783
7,James Harden,CLE,BKN,1,5.760078,0.335094,1.930167,0.0,1.930167,0.574763,0.574763,0.304430
8,Sam Merrill,CLE,BKN,1,5.415078,0.316933,1.716219,0.0,1.716219,0.511775,0.511775,0.247064
9,Jaylon Tyson,CLE,BKN,1,3.220533,0.332574,1.071066,0.0,1.071066,0.290363,0.290363,0.093827
